# Occupational classification · Interview research · Version 6

**Objective.** Produce a verifiable, end-to-end classification study for the simulated CBS survey, with explicit decisions for all 18 source columns, eight model representations and thirteen additional model and ensemble paths, group-aware validation, and separately evaluated contextual grouping.

**Source contract.** The source Excel workbook is loaded directly from the public repository or from a byte-identical local file specified through `CBS_OCCUPATION_WORKBOOK_PATH`. The SHA-256 fingerprint prevents silently analyzing a different dataset. The original workbook is never modified.

**Reproducibility.** This notebook runs on Google Colab with common scientific Python packages. All transformations that learn parameters are fitted inside the training portions of the respective evaluation splits. The notebook uses precise, self-describing code identifiers and separates steps into readable logical blocks.

In [ ]:
from __future__ import annotations

import hashlib
import io
import json
import os
import platform
import re
import urllib.request
import warnings

from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn

from IPython.display import display
from matplotlib.ticker import MaxNLocator
from scipy.sparse import issparse
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, calinski_harabasz_score, classification_report
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, normalize


ORIGINAL_WORKBOOK_PUBLIC_URL = "https://raw.githubusercontent.com/shulruferor-sudo/Advance-Data-Science/main/Datasets/cbs_occupation_survey_sample/DUMMY_DATASET.xlsx"
ORIGINAL_WORKBOOK_EXPECTED_SHA256 = "e1e7f391c0675dd14830eaae6b5cd114edc256f18b956bb48a454b0b1209e1ac"
OUTPUT_DIRECTORY = Path(os.environ.get("CBS_OCCUPATION_OUTPUT_DIRECTORY", "occupational_classification_v6_outputs"))
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

OCCUPATIONAL_DESCRIPTION_COLUMNS = ["SugAvoda", "ShemMachlaka", "SugMachlaka", "EzoAvoda", "TeurPeula", "TeurTafkid"]
NOMINAL_SURVEY_COLUMNS = ["MenahelEtMi", "MaamadAvoda", "MakorSachar", "TeudaGvoha"]
NUMERICAL_SURVEY_COLUMNS = ["shnotlimud"]
MODEL_INPUT_COLUMNS = OCCUPATIONAL_DESCRIPTION_COLUMNS + NOMINAL_SURVEY_COLUMNS + NUMERICAL_SURVEY_COLUMNS
PARTICIPANT_DESCRIPTION_SIGNATURE_COLUMNS = ["ShemAvoda", *OCCUPATIONAL_DESCRIPTION_COLUMNS]
OCCUPATION_CLASSIFICATION_TARGET_COLUMN = "SemelMishlachSofi"
NONINFORMATIVE_DESCRIPTION_VALUES = {'כנ"ל', 'כנ״ל', 'חסוי', 'לא יכול לפרט', '0'}

MAXIMUM_CONTEXTUAL_GROUPS_SAFETY_LIMIT = 1000
CONTEXTUAL_GROUPING_INITIALIZATIONS = 3
CONTEXTUAL_GROUPING_MAXIMUM_ITERATIONS = 150
MAXIMUM_CHARACTER_NGRAM_FEATURES = 900
REPRODUCIBLE_RANDOM_STATE = 42


source_workbook_local_path = os.environ.get("CBS_OCCUPATION_WORKBOOK_PATH")

if source_workbook_local_path:
    original_workbook_bytes = Path(source_workbook_local_path).read_bytes()
else:
    workbook_download_request = urllib.request.Request(ORIGINAL_WORKBOOK_PUBLIC_URL, headers={"User-Agent": "occupation-classification-interview-v6"})

    with urllib.request.urlopen(workbook_download_request, timeout=60) as downloaded_workbook:
        original_workbook_bytes = downloaded_workbook.read()

actual_source_workbook_sha256 = hashlib.sha256(original_workbook_bytes).hexdigest()
assert actual_source_workbook_sha256 == ORIGINAL_WORKBOOK_EXPECTED_SHA256, "Source dataset bytes differ from audited original"

original_survey_workbook = pd.ExcelFile(io.BytesIO(original_workbook_bytes), engine="openpyxl")
assert set(original_survey_workbook.sheet_names) == {"Survey Dataset", "Metadata"}

source_field_metadata = pd.read_excel(original_survey_workbook, sheet_name="Metadata", usecols="A:D", dtype="string")
source_field_metadata.columns = source_field_metadata.columns.astype(str).str.strip()
assert source_field_metadata.columns.tolist() == ["משתנה", "הסבר", "פורמט", "ערכים אפשריים"]

source_field_metadata = source_field_metadata.loc[source_field_metadata["משתנה"].notna()].copy()
source_field_metadata["משתנה"] = source_field_metadata["משתנה"].str.strip()
source_field_metadata["פורמט"] = source_field_metadata["פורמט"].str.strip()
assert source_field_metadata.shape == (18, 4)
assert source_field_metadata["משתנה"].is_unique
assert set(source_field_metadata["פורמט"]) <= {"INTEGER", "TEXT", "DATE"}

original_text_columns = source_field_metadata.loc[source_field_metadata["פורמט"].eq("TEXT"), "משתנה"].tolist()
original_survey_sheet = pd.read_excel(original_survey_workbook, sheet_name="Survey Dataset", dtype={field_name: "string" for field_name in original_text_columns})

participant_survey_records = original_survey_sheet.loc[original_survey_sheet["ID"].notna()].reset_index(drop=True)
nonparticipant_summary_rows = original_survey_sheet.loc[original_survey_sheet["ID"].isna()]

assert participant_survey_records.shape == (200, 18)
assert len(nonparticipant_summary_rows) == 1
assert set(source_field_metadata["משתנה"]) == set(participant_survey_records.columns)

print("Original workbook SHA-256:", actual_source_workbook_sha256)
print("Participant survey rows:", len(participant_survey_records))
print("Metadata fields:", len(source_field_metadata))
print("Nonparticipant summary rows excluded:", len(nonparticipant_summary_rows))
print("Python:", platform.python_version(), "| scikit-learn:", sklearn.__version__)

## All 18 source columns · Metadata-driven audit

Each original field is assessed against its own metadata-declared type, enumerations, code width, missingness, and explicit modeling role. Coded integers are interpreted as nominal categories where the metadata declares an enumeration. Partial occupation labels containing `X` are flagged for manual review, not treated as complete training targets.

The salary-source field is governed by the employment-status applicability relation documented in the source metadata. Other role decisions use operational leakage considerations rather than guessing from column data types alone.

In [ ]:
def infer_declared_enumerations_from_metadata(raw_enumeration_description):
    if pd.isna(raw_enumeration_description):
        return {}

    observed_code_descriptions = re.findall(r"(?m)^\s*(\d+)\.\s*(\S[^\n]*?)\s*$", str(raw_enumeration_description))

    return {int(code): description.strip() for code, description in observed_code_descriptions}


def infer_declared_code_width_from_metadata(raw_field_description):
    if pd.isna(raw_field_description):
        return None

    expected_code_width_match = re.search(r"מורכב\s*מ\s*[-־]?\s*(\d+)\s*ספרות", str(raw_field_description))

    return int(expected_code_width_match.group(1)) if expected_code_width_match else None


def identify_populated_source_values(source_field_values):
    return source_field_values.notna() & source_field_values.astype("string").str.strip().fillna("").ne("")


field_metadata_by_column = source_field_metadata.set_index("משתנה", verify_integrity=True)
declared_field_enumerations = {field_name: infer_declared_enumerations_from_metadata(field_metadata_by_column.loc[field_name, "ערכים אפשריים"]) for field_name in participant_survey_records.columns}
declared_field_code_widths = {field_name: infer_declared_code_width_from_metadata(field_metadata_by_column.loc[field_name, "הסבר"]) for field_name in participant_survey_records.columns}

assert declared_field_code_widths[OCCUPATION_CLASSIFICATION_TARGET_COLUMN] == 4
assert declared_field_code_widths["SemelAnafSofi"] == 4
assert set(OCCUPATIONAL_DESCRIPTION_COLUMNS) <= set(original_text_columns)
assert all(declared_field_enumerations[feature_name] for feature_name in NOMINAL_SURVEY_COLUMNS)
assert all(field_metadata_by_column.loc[feature_name, "פורמט"] == "INTEGER" for feature_name in NUMERICAL_SURVEY_COLUMNS)


source_column_audit_records = []
source_validation_issue_records = []

for field_name in participant_survey_records.columns:
    original_field_values = participant_survey_records[field_name]
    populated_field_values = identify_populated_source_values(original_field_values)
    expected_field_format = field_metadata_by_column.loc[field_name, "פורמט"]
    parsed_numeric_field_values = pd.to_numeric(original_field_values, errors="coerce") if expected_field_format == "INTEGER" else None
    field_validation_masks = {}

    if expected_field_format == "INTEGER":
        field_validation_masks["non_numeric"] = populated_field_values & parsed_numeric_field_values.isna()
        field_validation_masks["non_integral"] = parsed_numeric_field_values.notna() & parsed_numeric_field_values.mod(1).ne(0)

        if declared_field_enumerations[field_name]:
            field_validation_masks["outside_metadata_enumeration"] = parsed_numeric_field_values.notna() & ~parsed_numeric_field_values.isin(declared_field_enumerations[field_name])

    if expected_field_format == "TEXT" and declared_field_code_widths[field_name]:
        standardized_code_values = original_field_values.astype("string").str.strip()
        expected_code_shape = rf"[0-9Xx]{{{declared_field_code_widths[field_name]}}}"
        field_validation_masks["invalid_code_shape"] = populated_field_values & ~standardized_code_values.str.fullmatch(expected_code_shape, na=False)
        field_validation_masks["partially_classified_code"] = standardized_code_values.str.fullmatch(expected_code_shape, na=False) & standardized_code_values.str.contains("X", case=False, regex=False, na=False)

    if expected_field_format == "DATE":
        parsed_date_values = pd.to_datetime(original_field_values, errors="coerce", format="mixed")
        field_validation_masks["unparseable_date"] = populated_field_values & parsed_date_values.isna()

    for validation_rule_name, detected_issue_mask in field_validation_masks.items():
        if validation_rule_name == "partially_classified_code":
            continue

        for participant_row_number in participant_survey_records.index[detected_issue_mask.fillna(False)]:
            source_validation_issue_records.append({"participant_id": participant_survey_records.loc[participant_row_number, "ID"], "field_name": field_name, "validation_rule_name": validation_rule_name, "original_value": str(original_field_values.loc[participant_row_number])})

    declared_field_role = (
        "target_for_classification" if field_name == OCCUPATION_CLASSIFICATION_TARGET_COLUMN else
        "occupational_text_representation" if field_name in OCCUPATIONAL_DESCRIPTION_COLUMNS else
        "nominal_encoded_feature" if field_name in NOMINAL_SURVEY_COLUMNS else
        "numeric_imputed_feature" if field_name in NUMERICAL_SURVEY_COLUMNS else
        "identity_for_audit" if field_name == "ID" else
        "description_signature_only" if field_name == "ShemAvoda" else
        "excluded_to_prevent_unverified_operational_leakage" if field_name == "SemelAnafSofi" else
        "audit_only_not_approved_as_feature"
    )

    source_column_audit_records.append({
        "field_name": field_name,
        "metadata_declared_format": expected_field_format,
        "modeling_role": declared_field_role,
        "missing_values": int((~populated_field_values).sum()),
        "missing_percentage": float((~populated_field_values).mean() * 100),
        "distinct_populated_values": int(original_field_values.loc[populated_field_values].astype("string").nunique()),
        "metadata_enum_count": len(declared_field_enumerations[field_name]),
        "declared_code_width": declared_field_code_widths[field_name],
        "validation_issue_count": int(sum(detected_issue_mask.fillna(False).sum() for rule_name, detected_issue_mask in field_validation_masks.items() if rule_name != "partially_classified_code")),
        "partial_code_count": int(field_validation_masks["partially_classified_code"].sum()) if "partially_classified_code" in field_validation_masks else 0,
    })


source_column_audit = pd.DataFrame(source_column_audit_records)
source_validation_issues = pd.DataFrame(source_validation_issue_records)

assert len(source_column_audit) == 18
assert source_column_audit["field_name"].is_unique

applicable_salary_statuses = {code for code, meaning in declared_field_enumerations["MaamadAvoda"].items() if "שכיר" in meaning or "חבר קואופרטיב" in meaning}
assert applicable_salary_statuses == {1, 5}

reported_employment_status = pd.to_numeric(participant_survey_records["MaamadAvoda"], errors="coerce")
reported_salary_source_present = identify_populated_source_values(participant_survey_records["MakorSachar"])
nonapplicable_salary_source_conflicts = reported_employment_status.notna() & ~reported_employment_status.isin(applicable_salary_statuses) & reported_salary_source_present

print("Audited fields:", len(source_column_audit))
print("Detected type/domain violations:", len(source_validation_issues))
print("Employment/salary applicability conflicts:", int(nonapplicable_salary_source_conflicts.sum()))
display(source_column_audit[["field_name", "metadata_declared_format", "modeling_role", "missing_values", "distinct_populated_values", "validation_issue_count", "partial_code_count"]])


missingness_figure, missingness_axis = plt.subplots(figsize=(12.8, 7.0))
missingness_visual_data = source_column_audit.sort_values("missing_percentage")
missingness_role_colors = ["#12B8A6" if role in {"occupational_text_representation", "nominal_encoded_feature", "numeric_imputed_feature"} else "#64748B" for role in missingness_visual_data["modeling_role"]]

missingness_axis.barh(missingness_visual_data["field_name"], missingness_visual_data["missing_percentage"], color=missingness_role_colors)
missingness_axis.set_xlabel("Missing participant records (%)")
missingness_axis.set_xlim(0, 112)
missingness_axis.set_title("Source metadata audit · all 18 fields · 200 participant records", fontsize=15, weight="bold")
missingness_axis.grid(axis="x", alpha=0.15)
missingness_axis.set_axisbelow(True)

for visual_row_number, missing_percentage in enumerate(missingness_visual_data["missing_percentage"]):
    missingness_axis.text(missing_percentage + 1, visual_row_number, f"{missing_percentage:.0f}%", va="center", fontsize=9)

missingness_figure.tight_layout()
missingness_figure.savefig(OUTPUT_DIRECTORY / "all_18_metadata_fields_missingness.png", dpi=170, bbox_inches="tight")
plt.show()

## Feature preparation · Training-only learning · Leakage boundaries

The model uses six occupational text fields, four declared nominal fields, and one numerical education field. Identifiers, employer name, locality, start date, age, and final industry code remain outside the baseline feature set. In particular, availability of the final industry code before occupation classification is unverified.

A grouped external holdout is frozen before fitting representations. Internal evaluation uses ten disjoint group-aware folds derived only from the training set. Text-based groups are **proxies** for repeated descriptions, not verified participant identities, because stable respondent identifiers and submission timestamps are absent.

In [ ]:
def normalize_description_text_for_participant_grouping(raw_description):
    if pd.isna(raw_description):
        return ""

    standardized_description = re.sub(r"[^\w\u0590-\u05ff]+", " ", str(raw_description).lower())

    return " ".join(standardized_description.split())


def prepare_occupation_model_features(source_survey_records):
    prepared_model_inputs = pd.DataFrame(index=source_survey_records.index)

    for occupational_description_column in OCCUPATIONAL_DESCRIPTION_COLUMNS:
        cleaned_descriptions = source_survey_records[occupational_description_column].astype("string").fillna("").str.replace(r"\s+", " ", regex=True).str.strip()
        prepared_model_inputs[occupational_description_column] = cleaned_descriptions.mask(cleaned_descriptions.isin(NONINFORMATIVE_DESCRIPTION_VALUES), "").astype(str)

    for nominal_survey_column in NOMINAL_SURVEY_COLUMNS:
        numerical_nominal_codes = pd.to_numeric(source_survey_records[nominal_survey_column], errors="coerce").astype("Int64")
        prepared_model_inputs[nominal_survey_column] = numerical_nominal_codes.astype("string").fillna("MISSING").astype(str)

    employment_status_values = pd.to_numeric(source_survey_records["MaamadAvoda"], errors="coerce")
    employment_status_eligible_for_salary_source = employment_status_values.isin(applicable_salary_statuses)
    employment_status_unknown = employment_status_values.isna()

    prepared_model_inputs.loc[~employment_status_eligible_for_salary_source & ~employment_status_unknown, "MakorSachar"] = "NOT_APPLICABLE"
    prepared_model_inputs.loc[employment_status_unknown, "MakorSachar"] = "APPLICABILITY_UNKNOWN"

    for numerical_survey_column in NUMERICAL_SURVEY_COLUMNS:
        prepared_model_inputs[numerical_survey_column] = pd.to_numeric(source_survey_records[numerical_survey_column], errors="coerce")

    assert prepared_model_inputs.columns.tolist() == MODEL_INPUT_COLUMNS

    return prepared_model_inputs


original_occupation_codes = participant_survey_records[OCCUPATION_CLASSIFICATION_TARGET_COLUMN].astype("string").str.strip()
completely_labeled_records_mask = original_occupation_codes.str.fullmatch(r"\d{4}", na=False)
partially_labeled_records_mask = original_occupation_codes.str.fullmatch(r"[0-9Xx]{4}", na=False) & original_occupation_codes.str.contains("X", case=False, regex=False, na=False)

assert completely_labeled_records_mask.sum() == 191
assert partially_labeled_records_mask.sum() == 9
assert (completely_labeled_records_mask | partially_labeled_records_mask).all()

participant_description_signatures = participant_survey_records[PARTICIPANT_DESCRIPTION_SIGNATURE_COLUMNS].apply(
    lambda survey_record: "|".join(normalize_description_text_for_participant_grouping(survey_record[field_name]) for field_name in PARTICIPANT_DESCRIPTION_SIGNATURE_COLUMNS), axis=1
)

completely_labeled_participant_records = participant_survey_records.loc[completely_labeled_records_mask].copy()
partially_labeled_participant_records = participant_survey_records.loc[partially_labeled_records_mask].copy()

all_labeled_model_inputs = prepare_occupation_model_features(completely_labeled_participant_records)
all_labeled_occupation_codes = completely_labeled_participant_records[OCCUPATION_CLASSIFICATION_TARGET_COLUMN].astype(str)
all_labeled_description_groups = participant_description_signatures.loc[completely_labeled_participant_records.index]

external_holdout_group_splitter = GroupShuffleSplit(n_splits=1, test_size=0.22, random_state=REPRODUCIBLE_RANDOM_STATE)
training_row_positions, external_holdout_row_positions = next(external_holdout_group_splitter.split(all_labeled_model_inputs, all_labeled_occupation_codes, groups=all_labeled_description_groups))

training_model_inputs = all_labeled_model_inputs.iloc[training_row_positions]
external_holdout_model_inputs = all_labeled_model_inputs.iloc[external_holdout_row_positions]
training_occupation_codes = all_labeled_occupation_codes.iloc[training_row_positions]
external_holdout_occupation_codes = all_labeled_occupation_codes.iloc[external_holdout_row_positions]
training_description_groups = all_labeled_description_groups.iloc[training_row_positions]
external_holdout_description_groups = all_labeled_description_groups.iloc[external_holdout_row_positions]

assert len(training_model_inputs) == 149
assert len(external_holdout_model_inputs) == 42
assert set(training_description_groups).isdisjoint(set(external_holdout_description_groups))
assert OCCUPATION_CLASSIFICATION_TARGET_COLUMN not in training_model_inputs
assert "SemelAnafSofi" not in training_model_inputs

internal_validation_group_splitter = GroupKFold(n_splits=10)
internal_validation_group_partitions = list(internal_validation_group_splitter.split(training_model_inputs, training_occupation_codes, groups=training_description_groups))

for training_fold_positions, validation_fold_positions in internal_validation_group_partitions:
    assert set(training_description_groups.iloc[training_fold_positions]).isdisjoint(set(training_description_groups.iloc[validation_fold_positions]))

print("Training participant records:", len(training_model_inputs))
print("External holdout participant records:", len(external_holdout_model_inputs))
print("Unseen occupational target codes in holdout:", int((~external_holdout_occupation_codes.isin(set(training_occupation_codes))).sum()))
assert len(internal_validation_group_partitions) == 10
assert sorted(np.concatenate([validation_positions for _, validation_positions in internal_validation_group_partitions]).tolist()) == list(range(len(training_model_inputs)))

print("Disjoint grouped cross-validation folds:", len(internal_validation_group_partitions))

## Training-only text transformers · Explicit contextual grouping

One transformer generates direct character n-gram TF-IDF representations. Another trains a separate contextual grouping for each occupational text column, with explicit sparse or dense input storage and L2 normalization.

The number of groups is evaluated at every integer from two through `min(1000, number_of_distinct_training_descriptions - 1)`. Candidate scores use the Calinski–Harabasz variance ratio, which compares between-group dispersion against within-group dispersion. The optimal number of contextual groups means *the highest valid score among tested candidates*, not a universally optimal semantic or production grouping.

`KMeans` is the published scikit-learn estimator for **both sparse and dense inputs**, rather than a different library estimator named “SparseKMeans.” Smoothing is varied before the normalization step. Fitted encoders and groups never use target codes.

In [ ]:
def standardize_text_vectorizer_input(source_description_values):
    if isinstance(source_description_values, pd.DataFrame):
        selected_text_values = source_description_values.iloc[:, 0]
    elif isinstance(source_description_values, pd.Series):
        selected_text_values = source_description_values
    else:
        selected_text_values = pd.Series(np.asarray(source_description_values, dtype=object).reshape(-1))

    return selected_text_values.fillna("").astype(str).tolist()


class EmptySafeOccupationalTfidfEncoding(BaseEstimator, TransformerMixin):
    def __init__(self, l2_normalization_enabled=True, tfidf_smoothing_enabled=True, maximum_character_features=MAXIMUM_CHARACTER_NGRAM_FEATURES):
        self.l2_normalization_enabled = l2_normalization_enabled
        self.tfidf_smoothing_enabled = tfidf_smoothing_enabled
        self.maximum_character_features = maximum_character_features

    def fit(self, source_description_values, occupational_target_codes=None):
        training_descriptions = standardize_text_vectorizer_input(source_description_values)
        self.all_training_descriptions_empty_ = not any(len(text_value) >= 2 for text_value in training_descriptions)
        self.learned_text_vectorizer_ = None

        if not self.all_training_descriptions_empty_:
            self.learned_text_vectorizer_ = TfidfVectorizer(
                analyzer="char",
                ngram_range=(2, 4),
                max_features=self.maximum_character_features,
                norm="l2" if self.l2_normalization_enabled else None,
                smooth_idf=self.tfidf_smoothing_enabled,
                sublinear_tf=True,
            )

            self.learned_text_vectorizer_.fit(training_descriptions)

        return self

    def transform(self, source_description_values):
        prediction_descriptions = standardize_text_vectorizer_input(source_description_values)

        if self.all_training_descriptions_empty_:
            from scipy.sparse import csr_matrix

            return csr_matrix((len(prediction_descriptions), 1), dtype=float)

        return self.learned_text_vectorizer_.transform(prediction_descriptions)


class TrainingOnlyVarianceBalancedContextualGrouping(BaseEstimator, TransformerMixin):
    def __init__(self, occupational_description_columns=tuple(OCCUPATIONAL_DESCRIPTION_COLUMNS), tfidf_smoothing_enabled=True, vector_storage_format="sparse", maximum_contextual_groups=MAXIMUM_CONTEXTUAL_GROUPS_SAFETY_LIMIT, repeated_initializations=CONTEXTUAL_GROUPING_INITIALIZATIONS, maximum_iterations=CONTEXTUAL_GROUPING_MAXIMUM_ITERATIONS, random_state=REPRODUCIBLE_RANDOM_STATE):
        self.occupational_description_columns = occupational_description_columns
        self.tfidf_smoothing_enabled = tfidf_smoothing_enabled
        self.vector_storage_format = vector_storage_format
        self.maximum_contextual_groups = maximum_contextual_groups
        self.repeated_initializations = repeated_initializations
        self.maximum_iterations = maximum_iterations
        self.random_state = random_state

    def fit(self, source_description_frame, occupational_target_codes=None):
        if self.vector_storage_format not in {"dense", "sparse"}:
            raise ValueError("Only dense and sparse vector storage formats are supported")

        training_description_frame = pd.DataFrame(source_description_frame, columns=self.occupational_description_columns)
        self.trained_column_encoders_ = {}
        self.candidate_variance_balance_evaluations_ = []
        self.optimal_contextual_grouping_configurations_ = []
        self.optimal_contextual_group_memberships_ = []

        for occupational_text_column in self.occupational_description_columns:
            distinct_training_descriptions = sorted(description for description in training_description_frame[occupational_text_column].fillna("").astype(str).unique() if description.strip())
            number_of_distinct_descriptions = len(distinct_training_descriptions)

            trained_column_configuration = {
                "fallback_only": True,
                "known_description_group_assignments": {},
                "optimal_number_of_contextual_groups": None,
            }

            column_candidate_evaluations = []
            fitted_candidate_models = {}
            number_of_distinct_vector_representations = 0

            if number_of_distinct_descriptions >= 3:
                column_vectorizer = EmptySafeOccupationalTfidfEncoding(l2_normalization_enabled=True, tfidf_smoothing_enabled=self.tfidf_smoothing_enabled)
                normalized_sparse_description_vectors = column_vectorizer.fit_transform(distinct_training_descriptions).tocsr()
                normalized_dense_description_vectors = normalized_sparse_description_vectors.toarray()
                normalized_vector_norms = np.linalg.norm(normalized_dense_description_vectors, axis=1)

                np.testing.assert_allclose(normalized_vector_norms[normalized_vector_norms > 0], 1.0, rtol=1e-12, atol=1e-12)

                number_of_distinct_vector_representations = int(np.unique(np.round(normalized_dense_description_vectors, 12), axis=0).shape[0])
                actual_description_vectors = normalized_dense_description_vectors if self.vector_storage_format == "dense" else normalized_sparse_description_vectors
                maximum_tested_number_of_contextual_groups = min(self.maximum_contextual_groups, number_of_distinct_descriptions - 1)

                for number_of_contextual_groups in range(2, maximum_tested_number_of_contextual_groups + 1):
                    contextual_grouping_model = KMeans(n_clusters=number_of_contextual_groups, n_init=self.repeated_initializations, max_iter=self.maximum_iterations, random_state=self.random_state, algorithm="lloyd")
                    estimator_error_message = ""

                    try:
                        with warnings.catch_warnings(record=True) as convergence_warnings:
                            warnings.simplefilter("always", category=ConvergenceWarning)
                            observed_group_assignments = contextual_grouping_model.fit_predict(actual_description_vectors)

                        observed_number_of_groups = int(np.unique(observed_group_assignments).size)
                        estimator_warning_messages = " | ".join(str(warning.message) for warning in convergence_warnings)
                    except Exception as estimator_failure:
                        observed_group_assignments = np.array([], dtype=int)
                        observed_number_of_groups = 0
                        estimator_warning_messages = ""
                        estimator_error_message = f"{type(estimator_failure).__name__}: {estimator_failure}"

                    variance_balance_score = np.nan
                    within_group_dispersion = np.nan
                    evaluation_status = "valid"
                    observed_singleton_group_count = 0

                    if estimator_error_message:
                        evaluation_status = "estimator_failed"
                    elif observed_number_of_groups != number_of_contextual_groups:
                        evaluation_status = "fewer_nonempty_groups_than_requested"
                    else:
                        within_group_dispersion = float(contextual_grouping_model.inertia_)
                        observed_group_sizes = np.bincount(observed_group_assignments)
                        observed_singleton_group_count = int(np.count_nonzero(observed_group_sizes == 1))

                        if within_group_dispersion <= 1e-13:
                            evaluation_status = "zero_within_group_variance"
                        else:
                            variance_balance_score = float(calinski_harabasz_score(normalized_dense_description_vectors, observed_group_assignments))

                            if not np.isfinite(variance_balance_score):
                                evaluation_status = "nonfinite_variance_balance_score"
                            else:
                                fitted_candidate_models[number_of_contextual_groups] = contextual_grouping_model

                    candidate_evaluation = {
                        "occupational_text_column": occupational_text_column,
                        "tfidf_smoothing_enabled": self.tfidf_smoothing_enabled,
                        "vector_storage_format": self.vector_storage_format,
                        "number_of_distinct_text_values": number_of_distinct_descriptions,
                        "number_of_distinct_vector_representations": number_of_distinct_vector_representations,
                        "number_of_contextual_groups": number_of_contextual_groups,
                        "observed_number_of_contextual_groups": observed_number_of_groups,
                        "within_group_dispersion": within_group_dispersion,
                        "variance_balance_score": variance_balance_score,
                        "number_of_singleton_groups": observed_singleton_group_count,
                        "evaluation_status": evaluation_status,
                        "estimator_warning_messages": estimator_warning_messages,
                    }

                    column_candidate_evaluations.append(candidate_evaluation)
                    self.candidate_variance_balance_evaluations_.append(candidate_evaluation)

                valid_grouping_candidates = [candidate for candidate in column_candidate_evaluations if candidate["evaluation_status"] == "valid"]

                if valid_grouping_candidates:
                    optimal_grouping_candidate = sorted(valid_grouping_candidates, key=lambda candidate: (-candidate["variance_balance_score"], candidate["number_of_contextual_groups"]))[0]
                    optimal_number_of_contextual_groups = optimal_grouping_candidate["number_of_contextual_groups"]
                    selected_contextual_grouping_model = fitted_candidate_models[optimal_number_of_contextual_groups]

                    trained_column_configuration = {
                        "fallback_only": False,
                        "known_description_group_assignments": dict(zip(distinct_training_descriptions, selected_contextual_grouping_model.labels_)),
                        "optimal_number_of_contextual_groups": optimal_number_of_contextual_groups,
                        "selected_contextual_grouping_model": selected_contextual_grouping_model,
                        "learned_text_vectorizer": column_vectorizer,
                    }

                    for original_training_description, selected_group_number in zip(distinct_training_descriptions, selected_contextual_grouping_model.labels_):
                        self.optimal_contextual_group_memberships_.append({
                            "occupational_text_column": occupational_text_column,
                            "tfidf_smoothing_enabled": self.tfidf_smoothing_enabled,
                            "vector_storage_format": self.vector_storage_format,
                            "observed_description": original_training_description,
                            "assigned_contextual_group": f"contextual_group_{int(selected_group_number):03d}",
                        })

            self.trained_column_encoders_[occupational_text_column] = trained_column_configuration
            self.optimal_contextual_grouping_configurations_.append({
                "occupational_text_column": occupational_text_column,
                "tfidf_smoothing_enabled": self.tfidf_smoothing_enabled,
                "vector_storage_format": self.vector_storage_format,
                "number_of_distinct_text_values": number_of_distinct_descriptions,
                "number_of_candidate_group_counts": len(column_candidate_evaluations),
                "number_of_valid_candidate_group_counts": sum(candidate["evaluation_status"] == "valid" for candidate in column_candidate_evaluations),
                "optimal_number_of_contextual_groups": trained_column_configuration["optimal_number_of_contextual_groups"],
                "optimal_variance_balance_score": optimal_grouping_candidate["variance_balance_score"] if not trained_column_configuration["fallback_only"] else np.nan,
                "optimal_singleton_group_count": optimal_grouping_candidate["number_of_singleton_groups"] if not trained_column_configuration["fallback_only"] else np.nan,
            })

        return self

    def transform(self, source_description_frame):
        prediction_description_frame = pd.DataFrame(source_description_frame, columns=self.occupational_description_columns)
        categorical_grouping_features = np.empty((len(prediction_description_frame), len(self.occupational_description_columns)), dtype=object)

        for text_column_position, occupational_text_column in enumerate(self.occupational_description_columns):
            trained_column_configuration = self.trained_column_encoders_[occupational_text_column]
            prediction_descriptions = prediction_description_frame[occupational_text_column].fillna("").astype(str).tolist()
            predicted_contextual_groups = []

            for occupational_description in prediction_descriptions:
                if not occupational_description.strip():
                    predicted_contextual_groups.append("MISSING_DESCRIPTION")
                elif occupational_description in trained_column_configuration["known_description_group_assignments"]:
                    existing_group_number = int(trained_column_configuration["known_description_group_assignments"][occupational_description])
                    predicted_contextual_groups.append(f"contextual_group_{existing_group_number:03d}")
                elif trained_column_configuration["fallback_only"]:
                    predicted_contextual_groups.append("UNSEEN_DESCRIPTION")
                else:
                    unseen_description_vector = trained_column_configuration["learned_text_vectorizer"].transform([occupational_description])
                    unseen_description_group = int(trained_column_configuration["selected_contextual_grouping_model"].predict(unseen_description_vector if self.vector_storage_format == "sparse" else unseen_description_vector.toarray())[0])
                    predicted_contextual_groups.append(f"contextual_group_{unseen_description_group:03d}")

            categorical_grouping_features[:, text_column_position] = predicted_contextual_groups

        return categorical_grouping_features

    def get_feature_names_out(self, input_features=None):
        return np.asarray(self.occupational_description_columns, dtype=object)


print("Eight representation experiments use the same source-column contracts and classifier")

## Four direct encodings + four contextual grouping configurations

The **four direct encodings** are one-hot of whole description labels, direct unnormalized and unsmoothed TF-IDF, L2-normalized unsmoothed TF-IDF, and L2-normalized smoothed TF-IDF.

The **four contextual grouping configurations** use L2-normalized TF-IDF with dense or sparse `KMeans` inputs and smoothed or unsmoothed inverse-document-frequency. The selected group identifiers are nominally encoded before classification.

Every representation uses exactly the same six text columns, four nominal columns, numeric education preprocessing, and logistic regression classifier.

In [ ]:
EVALUATED_OCCUPATIONAL_TEXT_REPRESENTATIONS = [
    {"representation_name": "01 · One-hot description labels", "representation_family": "one_hot", "l2_normalization_enabled": False, "tfidf_smoothing_enabled": None, "vector_storage_format": None},
    {"representation_name": "02 · Direct TF-IDF · no L2 · unsmoothed", "representation_family": "direct_tfidf", "l2_normalization_enabled": False, "tfidf_smoothing_enabled": False, "vector_storage_format": "sparse"},
    {"representation_name": "03 · Direct TF-IDF · L2 · unsmoothed", "representation_family": "direct_tfidf", "l2_normalization_enabled": True, "tfidf_smoothing_enabled": False, "vector_storage_format": "sparse"},
    {"representation_name": "04 · Direct TF-IDF · L2 · smoothed", "representation_family": "direct_tfidf", "l2_normalization_enabled": True, "tfidf_smoothing_enabled": True, "vector_storage_format": "sparse"},
    {"representation_name": "05 · Contextual groups · dense · unsmoothed", "representation_family": "contextual_grouping", "l2_normalization_enabled": True, "tfidf_smoothing_enabled": False, "vector_storage_format": "dense"},
    {"representation_name": "06 · Contextual groups · sparse · unsmoothed", "representation_family": "contextual_grouping", "l2_normalization_enabled": True, "tfidf_smoothing_enabled": False, "vector_storage_format": "sparse"},
    {"representation_name": "07 · Contextual groups · dense · smoothed", "representation_family": "contextual_grouping", "l2_normalization_enabled": True, "tfidf_smoothing_enabled": True, "vector_storage_format": "dense"},
    {"representation_name": "08 · Contextual groups · sparse · smoothed", "representation_family": "contextual_grouping", "l2_normalization_enabled": True, "tfidf_smoothing_enabled": True, "vector_storage_format": "sparse"},
]

assert len(EVALUATED_OCCUPATIONAL_TEXT_REPRESENTATIONS) == 8


def build_training_only_occupation_classification_pipeline(representation_configuration):
    representation_family = representation_configuration["representation_family"]

    if representation_family == "one_hot":
        occupational_description_transformers = [("occupational_description_one_hot_encoding", OneHotEncoder(handle_unknown="ignore"), OCCUPATIONAL_DESCRIPTION_COLUMNS)]

    elif representation_family == "direct_tfidf":
        occupational_description_transformers = [
            (
                f"direct_tfidf_encoding_of_{occupational_text_column}",
                EmptySafeOccupationalTfidfEncoding(
                    l2_normalization_enabled=representation_configuration["l2_normalization_enabled"],
                    tfidf_smoothing_enabled=representation_configuration["tfidf_smoothing_enabled"],
                ),
                occupational_text_column,
            )
            for occupational_text_column in OCCUPATIONAL_DESCRIPTION_COLUMNS
        ]

    elif representation_family == "contextual_grouping":
        training_only_contextual_grouping = TrainingOnlyVarianceBalancedContextualGrouping(
            occupational_description_columns=tuple(OCCUPATIONAL_DESCRIPTION_COLUMNS),
            tfidf_smoothing_enabled=representation_configuration["tfidf_smoothing_enabled"],
            vector_storage_format=representation_configuration["vector_storage_format"],
        )

        occupational_description_transformers = [
            (
                "selected_contextual_group_labels_then_one_hot",
                Pipeline([
                    ("optimal_contextual_grouping_by_variance_balance", training_only_contextual_grouping),
                    ("nominal_encoding_of_contextual_group_labels", OneHotEncoder(handle_unknown="ignore")),
                ]),
                OCCUPATIONAL_DESCRIPTION_COLUMNS,
            )
        ]

    else:
        raise ValueError("Unknown occupational text representation")

    survey_feature_transformer = ColumnTransformer(
        [
            *occupational_description_transformers,
            ("source_nominal_code_one_hot_encoding", OneHotEncoder(handle_unknown="ignore"), NOMINAL_SURVEY_COLUMNS),
            (
                "numerical_source_feature_preprocessing",
                Pipeline([
                    ("training_median_imputation", SimpleImputer(strategy="median", add_indicator=True)),
                    ("training_only_standardization", StandardScaler()),
                ]),
                NUMERICAL_SURVEY_COLUMNS,
            ),
        ],
        remainder="drop",
        sparse_threshold=1.0,
    )

    occupational_classifier = LogisticRegression(C=1.0, solver="lbfgs", max_iter=1200, random_state=REPRODUCIBLE_RANDOM_STATE)

    return Pipeline([
        ("train_only_survey_representation", survey_feature_transformer),
        ("four_digit_occupation_code_classifier", occupational_classifier),
    ])


def measure_occupation_classification_performance(actual_occupation_codes, predicted_occupation_codes):
    return {
        "accuracy": float(accuracy_score(actual_occupation_codes, predicted_occupation_codes)),
        "macro_precision": float(precision_score(actual_occupation_codes, predicted_occupation_codes, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(actual_occupation_codes, predicted_occupation_codes, average="macro", zero_division=0)),
        "macro_f1": float(f1_score(actual_occupation_codes, predicted_occupation_codes, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(actual_occupation_codes, predicted_occupation_codes, average="weighted", zero_division=0)),
    }


for representation_configuration in EVALUATED_OCCUPATIONAL_TEXT_REPRESENTATIONS:
    print(representation_configuration["representation_name"])

## Group-aware classification evaluation · Eight representation paths, ten grouped folds

Each representation is learned anew inside each of ten nonoverlapping participant-description-group validation folds. Every fold and every out-of-fold prediction is preserved. Selection uses the mean **macro-F1**, with macro-recall and a predeclared complexity order resolving ties. The 42-record external holdout is evaluated after the internal selection is fixed. Other holdout scores are shown as transparent comparisons and are not used to revise the selected configuration.

The notebook reports accuracy, macro-precision, macro-recall, macro-F1, weighted-F1, exact correct counts, fitted feature dimensions, and training runtime. These are simulation results rather than estimates of operational accuracy.

In [ ]:
internal_validation_performance_records = []
all_eight_representation_out_of_fold_predictions = []

for validation_partition_number, (training_fold_positions, validation_fold_positions) in enumerate(internal_validation_group_partitions, start=1):
    fold_training_inputs = training_model_inputs.iloc[training_fold_positions]
    fold_validation_inputs = training_model_inputs.iloc[validation_fold_positions]
    fold_training_occupation_codes = training_occupation_codes.iloc[training_fold_positions]
    fold_validation_occupation_codes = training_occupation_codes.iloc[validation_fold_positions]

    for representation_configuration in EVALUATED_OCCUPATIONAL_TEXT_REPRESENTATIONS:
        occupation_classification_pipeline = build_training_only_occupation_classification_pipeline(representation_configuration)
        training_start_time = perf_counter()

        occupation_classification_pipeline.fit(fold_training_inputs, fold_training_occupation_codes)
        predicted_validation_occupation_codes = occupation_classification_pipeline.predict(fold_validation_inputs)

        recorded_validation_performance = {
            "internal_validation_partition": validation_partition_number,
            "representation_name": representation_configuration["representation_name"],
            "representation_family": representation_configuration["representation_family"],
            "training_participant_records": len(fold_training_inputs),
            "validation_participant_records": len(fold_validation_inputs),
            "unseen_occupation_code_records": int((~fold_validation_occupation_codes.isin(set(fold_training_occupation_codes))).sum()),
            "training_elapsed_seconds": perf_counter() - training_start_time,
        }

        recorded_validation_performance.update(measure_occupation_classification_performance(fold_validation_occupation_codes, predicted_validation_occupation_codes))
        internal_validation_performance_records.append(recorded_validation_performance)

        for participant_position, true_occupation_code, predicted_occupation_code in zip(validation_fold_positions, fold_validation_occupation_codes, predicted_validation_occupation_codes):
            all_eight_representation_out_of_fold_predictions.append({
                "cross_validation_fold": validation_partition_number,
                "representation_name": representation_configuration["representation_name"],
                "source_row_position": int(participant_position),
                "participant_id": int(completely_labeled_participant_records.iloc[training_row_positions[participant_position]]["ID"]),
                "actual_occupation_code": str(true_occupation_code),
                "predicted_occupation_code": str(predicted_occupation_code),
                "prediction_correct": bool(true_occupation_code == predicted_occupation_code),
            })

    print(f"Completed internal validation partition {validation_partition_number} of {len(internal_validation_group_partitions)}", flush=True)


internal_validation_performance = pd.DataFrame(internal_validation_performance_records)
all_eight_representation_out_of_fold_predictions = pd.DataFrame(all_eight_representation_out_of_fold_predictions)

assert len(all_eight_representation_out_of_fold_predictions) == len(EVALUATED_OCCUPATIONAL_TEXT_REPRESENTATIONS) * len(training_model_inputs)
assert all_eight_representation_out_of_fold_predictions.groupby("representation_name")["source_row_position"].nunique().eq(len(training_model_inputs)).all()

pooled_out_of_fold_scores_for_each_representation = []

for representation_configuration in EVALUATED_OCCUPATIONAL_TEXT_REPRESENTATIONS:
    representation_out_of_fold_records = all_eight_representation_out_of_fold_predictions.loc[all_eight_representation_out_of_fold_predictions["representation_name"].eq(representation_configuration["representation_name"])]
    pooled_scores = measure_occupation_classification_performance(representation_out_of_fold_records["actual_occupation_code"], representation_out_of_fold_records["predicted_occupation_code"])
    pooled_out_of_fold_scores_for_each_representation.append({"representation_name": representation_configuration["representation_name"], **{f"pooled_out_of_fold_{metric}": score for metric, score in pooled_scores.items()}})

pooled_out_of_fold_scores_for_each_representation = pd.DataFrame(pooled_out_of_fold_scores_for_each_representation)
internal_representation_summary = (
    internal_validation_performance.groupby(["representation_name", "representation_family"], as_index=False)
    .agg(
        internal_macro_f1_mean=("macro_f1", "mean"),
        internal_macro_f1_standard_deviation=("macro_f1", "std"),
        internal_macro_recall_mean=("macro_recall", "mean"),
        internal_accuracy_mean=("accuracy", "mean"),
        cumulative_training_elapsed_seconds=("training_elapsed_seconds", "sum"),
    )
)

representation_complexity_order = {configuration["representation_name"]: representation_number for representation_number, configuration in enumerate(EVALUATED_OCCUPATIONAL_TEXT_REPRESENTATIONS)}
internal_representation_summary["representation_complexity_order"] = internal_representation_summary["representation_name"].map(representation_complexity_order)

internal_representation_summary = internal_representation_summary.merge(pooled_out_of_fold_scores_for_each_representation, on="representation_name", how="left", validate="one_to_one")

internal_representation_summary = internal_representation_summary.sort_values(
    ["internal_macro_f1_mean", "internal_macro_recall_mean", "representation_complexity_order"], ascending=[False, False, True]
).reset_index(drop=True)

internally_selected_representation_name = internal_representation_summary.loc[0, "representation_name"]

print("Selected by internal grouped validation only:", internally_selected_representation_name)
display(internal_representation_summary.round(4))


external_holdout_performance_records = []
external_holdout_predicted_codes_by_representation = {}
fully_trained_classification_pipelines = {}

for representation_configuration in EVALUATED_OCCUPATIONAL_TEXT_REPRESENTATIONS:
    representation_name = representation_configuration["representation_name"]
    occupation_classification_pipeline = build_training_only_occupation_classification_pipeline(representation_configuration)
    training_start_time = perf_counter()

    occupation_classification_pipeline.fit(training_model_inputs, training_occupation_codes)
    external_holdout_prediction = occupation_classification_pipeline.predict(external_holdout_model_inputs)
    transformed_training_input_dimensions = occupation_classification_pipeline.named_steps["train_only_survey_representation"].transform(training_model_inputs).shape[1]

    observed_holdout_performance = {
        "representation_name": representation_name,
        "representation_family": representation_configuration["representation_family"],
        "correctly_classified_participant_records": int(np.sum(external_holdout_prediction == external_holdout_occupation_codes.to_numpy())),
        "total_transformed_input_features": int(transformed_training_input_dimensions),
        "training_elapsed_seconds": perf_counter() - training_start_time,
    }

    observed_holdout_performance.update(measure_occupation_classification_performance(external_holdout_occupation_codes, external_holdout_prediction))
    external_holdout_performance_records.append(observed_holdout_performance)
    external_holdout_predicted_codes_by_representation[representation_name] = external_holdout_prediction
    fully_trained_classification_pipelines[representation_name] = occupation_classification_pipeline

    print("Completed frozen holdout assessment:", representation_name, flush=True)


external_holdout_performance = pd.DataFrame(external_holdout_performance_records)
external_holdout_performance["internally_selected"] = external_holdout_performance["representation_name"].eq(internally_selected_representation_name)

external_holdout_predictions = pd.DataFrame({
    "participant_id": completely_labeled_participant_records.iloc[external_holdout_row_positions]["ID"].astype(int).to_numpy(),
    "actual_occupation_code": external_holdout_occupation_codes.to_numpy(),
    **external_holdout_predicted_codes_by_representation,
})

assert len(external_holdout_performance) == 8
assert len(external_holdout_predictions) == 42

print("Frozen holdout comparisons across eight representations:")
display(external_holdout_performance.round(4))

## Every contextual grouping candidate · Actual variance-balance curves

Four curves are plotted for **each of six original occupational description columns**, using the identical candidate group-count sequence within that column. Every measured score is exported to CSV, with all four evaluated configurations represented and the selected scores annotated.

The horizontal axes show integer numbers of contextual groups and the vertical axes show integer score ticks. Numeric labels preserve decimal score precision at selected optima. Full candidate tables retain unrounded scores.

**Interpretive warning.** Optimizing Calinski–Harabasz may produce almost one group per distinct description. Inspect the number of singleton groups before concluding that the optimization created useful contextual consolidation.

In [ ]:
all_contextual_grouping_candidate_evaluations = []
all_optimal_contextual_grouping_configurations = []
all_optimal_group_memberships = []

for representation_configuration in EVALUATED_OCCUPATIONAL_TEXT_REPRESENTATIONS:
    if representation_configuration["representation_family"] != "contextual_grouping":
        continue

    matching_fitted_pipeline = fully_trained_classification_pipelines[representation_configuration["representation_name"]]
    matched_survey_encoder = matching_fitted_pipeline.named_steps["train_only_survey_representation"]
    selected_grouping_preprocessor = matched_survey_encoder.named_transformers_["selected_contextual_group_labels_then_one_hot"]
    selected_contextual_grouping_transformer = selected_grouping_preprocessor.named_steps["optimal_contextual_grouping_by_variance_balance"]

    all_contextual_grouping_candidate_evaluations.extend(selected_contextual_grouping_transformer.candidate_variance_balance_evaluations_)
    all_optimal_contextual_grouping_configurations.extend(selected_contextual_grouping_transformer.optimal_contextual_grouping_configurations_)
    all_optimal_group_memberships.extend(selected_contextual_grouping_transformer.optimal_contextual_group_memberships_)


contextual_grouping_candidate_evaluations = pd.DataFrame(all_contextual_grouping_candidate_evaluations)
optimal_contextual_grouping_configurations = pd.DataFrame(all_optimal_contextual_grouping_configurations)
observed_optimal_contextual_group_memberships = pd.DataFrame(all_optimal_group_memberships)

assert len(optimal_contextual_grouping_configurations) == 24
assert len(contextual_grouping_candidate_evaluations) == 920
assert (contextual_grouping_candidate_evaluations["evaluation_status"] == "valid").all()

for occupational_text_column in OCCUPATIONAL_DESCRIPTION_COLUMNS:
    per_column_candidates = contextual_grouping_candidate_evaluations.loc[contextual_grouping_candidate_evaluations["occupational_text_column"].eq(occupational_text_column)]
    all_evaluated_group_sequences = [grouping_partition["number_of_contextual_groups"].tolist() for _, grouping_partition in per_column_candidates.groupby(["tfidf_smoothing_enabled", "vector_storage_format"])]
    declared_number_of_distinct_descriptions = int(per_column_candidates["number_of_distinct_text_values"].iloc[0])
    required_candidate_group_counts = list(range(2, min(MAXIMUM_CONTEXTUAL_GROUPS_SAFETY_LIMIT, declared_number_of_distinct_descriptions - 1) + 1))

    assert len(all_evaluated_group_sequences) == 4
    assert all(actual_group_sequence == required_candidate_group_counts for actual_group_sequence in all_evaluated_group_sequences)


contextual_grouping_legend_definitions = [
    (False, "dense", "No IDF smoothing · dense", "#F59E0B", "-"),
    (False, "sparse", "No IDF smoothing · sparse", "#F472B6", "--"),
    (True, "dense", "Smoothed IDF · dense", "#22D3EE", "-"),
    (True, "sparse", "Smoothed IDF · sparse", "#A78BFA", "--"),
]


def draw_contextual_grouping_variance_balance_curves(occupational_text_columns_to_display, figure_filename, number_of_plot_columns):
    number_of_plot_rows = int(np.ceil(len(occupational_text_columns_to_display) / number_of_plot_columns))
    comparison_figure, contextual_grouping_plot_axes = plt.subplots(number_of_plot_rows, number_of_plot_columns, figsize=(7.0 * number_of_plot_columns, 5.5 * number_of_plot_rows), squeeze=False)

    for occupational_text_column, contextual_grouping_plot_axis in zip(occupational_text_columns_to_display, contextual_grouping_plot_axes.flat):
        text_column_candidates = contextual_grouping_candidate_evaluations.loc[contextual_grouping_candidate_evaluations["occupational_text_column"].eq(occupational_text_column)]

        for smoothing_enabled, storage_format, configuration_label, plotted_line_color, plotted_line_style in contextual_grouping_legend_definitions:
            matching_configuration_candidates = text_column_candidates.loc[text_column_candidates["tfidf_smoothing_enabled"].eq(smoothing_enabled) & text_column_candidates["vector_storage_format"].eq(storage_format)].sort_values("number_of_contextual_groups")
            matching_configuration_optimum = optimal_contextual_grouping_configurations.loc[optimal_contextual_grouping_configurations["occupational_text_column"].eq(occupational_text_column) & optimal_contextual_grouping_configurations["tfidf_smoothing_enabled"].eq(smoothing_enabled) & optimal_contextual_grouping_configurations["vector_storage_format"].eq(storage_format)].iloc[0]

            contextual_grouping_plot_axis.plot(matching_configuration_candidates["number_of_contextual_groups"], matching_configuration_candidates["variance_balance_score"], color=plotted_line_color, linestyle=plotted_line_style, linewidth=2.1, alpha=0.95, label=configuration_label)

            optimal_number_of_contextual_groups = int(matching_configuration_optimum["optimal_number_of_contextual_groups"])
            optimal_variance_balance_score = float(matching_configuration_optimum["optimal_variance_balance_score"])

            contextual_grouping_plot_axis.scatter(optimal_number_of_contextual_groups, optimal_variance_balance_score, color=plotted_line_color, edgecolor="white", s=65, linewidth=1.1, zorder=6)
            contextual_grouping_plot_axis.annotate(f"{optimal_variance_balance_score:.2f}", (optimal_number_of_contextual_groups, optimal_variance_balance_score), xytext=(-5, 11 if storage_format == "dense" else -18), textcoords="offset points", fontsize=8, color=plotted_line_color, ha="right", weight="bold")

        contextual_grouping_plot_axis.set_title(occupational_text_column, fontsize=15, weight="bold")
        contextual_grouping_plot_axis.set_xlabel("Number of contextual groups", fontsize=10)
        contextual_grouping_plot_axis.set_ylabel("Variance balance score", fontsize=10)
        contextual_grouping_plot_axis.xaxis.set_major_locator(MaxNLocator(integer=True, nbins=7))
        contextual_grouping_plot_axis.yaxis.set_major_locator(MaxNLocator(integer=True, nbins=7))
        contextual_grouping_plot_axis.set_ylim(bottom=0)
        contextual_grouping_plot_axis.grid(alpha=0.18)
        contextual_grouping_plot_axis.spines[["top", "right"]].set_visible(False)

    visible_plot_axes = list(contextual_grouping_plot_axes.flat)

    for unused_plot_axis in visible_plot_axes[len(occupational_text_columns_to_display):]:
        unused_plot_axis.set_visible(False)

    legend_handles, legend_names = visible_plot_axes[0].get_legend_handles_labels()
    comparison_figure.legend(legend_handles, legend_names, ncol=4, loc="lower center", fontsize=11)
    comparison_figure.suptitle("Training-only contextual grouping · all candidate group counts · variance-balance comparison", fontsize=17, fontweight="bold")
    comparison_figure.tight_layout(rect=(0, 0.06, 1, 0.94), h_pad=2.7, w_pad=2.0)
    comparison_figure.savefig(OUTPUT_DIRECTORY / figure_filename, dpi=180, bbox_inches="tight")
    plt.show()


draw_contextual_grouping_variance_balance_curves(OCCUPATIONAL_DESCRIPTION_COLUMNS, "four_clustering_configs_all_six_text_columns.png", number_of_plot_columns=3)
draw_contextual_grouping_variance_balance_curves(OCCUPATIONAL_DESCRIPTION_COLUMNS[:3], "four_clustering_configs_first_three_text_columns.png", number_of_plot_columns=3)
draw_contextual_grouping_variance_balance_curves(OCCUPATIONAL_DESCRIPTION_COLUMNS[3:], "four_clustering_configs_last_three_text_columns.png", number_of_plot_columns=3)


optimal_grouping_summary_for_display = optimal_contextual_grouping_configurations.copy()
optimal_grouping_summary_for_display["fraction_of_groups_containing_one_description"] = optimal_grouping_summary_for_display["optimal_singleton_group_count"] / optimal_grouping_summary_for_display["optimal_number_of_contextual_groups"]

print("Complete candidate evaluations:", len(contextual_grouping_candidate_evaluations))
print("Mathematically invalid candidate evaluations:", int(contextual_grouping_candidate_evaluations["evaluation_status"].ne("valid").sum()))
display(optimal_grouping_summary_for_display.round(4))

## Graphical performance comparison · Validation, holdout, dimensions, cost

The following charts display the observed macro metrics, internal-to-external comparison, transformed feature dimensions, and observed execution time for all eight representations. Model-selection statements must use internal validation only.

In [ ]:
representation_plot_order = [configuration["representation_name"] for configuration in EVALUATED_OCCUPATIONAL_TEXT_REPRESENTATIONS]
internal_metric_plot_data = internal_representation_summary.set_index("representation_name").loc[representation_plot_order]
external_metric_plot_data = external_holdout_performance.set_index("representation_name").loc[representation_plot_order]
readable_representation_axis_labels = [representation_name.split(" · ", maxsplit=1)[-1] for representation_name in representation_plot_order]


heldout_metric_figure, heldout_metric_axis = plt.subplots(figsize=(14, 7.0))
representation_horizontal_positions = np.arange(len(representation_plot_order))

heldout_metric_axis.bar(representation_horizontal_positions - 0.18, external_metric_plot_data["macro_f1"], width=0.35, label="Held-out macro-F1", color="#22D3EE")
heldout_metric_axis.bar(representation_horizontal_positions + 0.18, external_metric_plot_data["macro_recall"], width=0.35, label="Held-out macro-recall", color="#F59E0B")
heldout_metric_axis.set_xticks(representation_horizontal_positions, readable_representation_axis_labels, rotation=24, ha="right")
heldout_metric_axis.set_ylim(0, max(0.55, float(external_metric_plot_data[["macro_f1", "macro_recall"]].max().max()) + 0.13))
heldout_metric_axis.set_ylabel("Held-out classification score")
heldout_metric_axis.set_title("Eight text representations · identical classifier and external holdout", weight="bold", fontsize=15)
heldout_metric_axis.grid(axis="y", alpha=0.16)
heldout_metric_axis.set_axisbelow(True)
heldout_metric_axis.legend(frameon=False)

for plotted_bar in heldout_metric_axis.patches:
    heldout_metric_axis.annotate(f"{plotted_bar.get_height():.3f}", (plotted_bar.get_x() + plotted_bar.get_width() / 2, plotted_bar.get_height()), xytext=(0, 4), textcoords="offset points", ha="center", fontsize=8)

heldout_metric_figure.tight_layout()
heldout_metric_figure.savefig(OUTPUT_DIRECTORY / "eight_representations_heldout_macro_metrics.png", dpi=180, bbox_inches="tight")
plt.show()


comparison_metric_figure, comparison_metric_axes = plt.subplots(1, 2, figsize=(15, 6.4))

comparison_metric_axes[0].barh(readable_representation_axis_labels[::-1], internal_metric_plot_data["internal_macro_f1_mean"].to_numpy()[::-1], color="#38BDF8")
comparison_metric_axes[0].set_xlabel("Mean grouped-validation macro-F1")
comparison_metric_axes[0].set_title("Internal selection evidence", weight="bold")
comparison_metric_axes[0].set_xlim(0, max(0.5, float(internal_metric_plot_data["internal_macro_f1_mean"].max()) + 0.08))
comparison_metric_axes[0].grid(axis="x", alpha=0.15)

comparison_metric_axes[1].barh(readable_representation_axis_labels[::-1], external_metric_plot_data["accuracy"].to_numpy()[::-1], color="#A78BFA")
comparison_metric_axes[1].set_xlabel("External holdout accuracy")
comparison_metric_axes[1].set_title("Frozen holdout · descriptive only", weight="bold")
comparison_metric_axes[1].set_xlim(0, 1)
comparison_metric_axes[1].grid(axis="x", alpha=0.15)

for comparison_metric_axis in comparison_metric_axes:
    for metric_bar in comparison_metric_axis.patches:
        comparison_metric_axis.text(metric_bar.get_width() + 0.008, metric_bar.get_y() + metric_bar.get_height() / 2, f"{metric_bar.get_width():.3f}", va="center", fontsize=9)

comparison_metric_figure.tight_layout(w_pad=3)
comparison_metric_figure.savefig(OUTPUT_DIRECTORY / "internal_validation_versus_external_holdout.png", dpi=180, bbox_inches="tight")
plt.show()


representation_cost_figure, representation_cost_axes = plt.subplots(1, 2, figsize=(15, 5.6))
representation_cost_axes[0].barh(readable_representation_axis_labels[::-1], external_metric_plot_data["total_transformed_input_features"].to_numpy()[::-1], color="#14B8A6")
representation_cost_axes[0].set_xlabel("Transformed training feature dimensions")
representation_cost_axes[0].set_title("Representation dimensionality", weight="bold")
representation_cost_axes[0].grid(axis="x", alpha=0.15)

representation_cost_axes[1].barh(readable_representation_axis_labels[::-1], external_metric_plot_data["training_elapsed_seconds"].to_numpy()[::-1], color="#FB923C")
representation_cost_axes[1].set_xlabel("Observed training time (seconds)")
representation_cost_axes[1].set_title("Execution cost · this environment only", weight="bold")
representation_cost_axes[1].grid(axis="x", alpha=0.15)

representation_cost_figure.tight_layout(w_pad=3)
representation_cost_figure.savefig(OUTPUT_DIRECTORY / "eight_representations_feature_dimensions_and_cost.png", dpi=180, bbox_inches="tight")
plt.show()

## Verified artifacts · Partial codes for mandatory human review

All source field audits, candidate scores, selected group counts, group memberships, internal evaluation partitions, holdout metrics and individual predictions are exported as reproducible CSV files.

After the evaluation is complete, the **internally selected** representation is refitted using the 191 complete labels to generate suggestions for nine partially labeled records. Those suggestions are **not** verified occupation codes and require human coder review.

### Methodological limitations and independent sources

- Original records are synthetic and manually labeled, with no independent ground-truth corrections.
- Source `ID` is a row number, not a verified persistent respondent identifier. No timestamp-based repetition control is possible.
- The chosen variance-balance score is a mathematical clustering diagnostic, not occupational-semantic validation. Singleton-dominant solutions remain a real failure mode.
- Unseen occupational labels in holdout and low-support classes limit macro metric stability.
- Production requires validated participant linkage, longitudinal splits, independent labels, human review, and drift monitoring.

**Official documentation:** [scikit-learn KMeans](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html), [Calinski–Harabasz score](https://scikit-learn.org/stable/modules/clustering.html#calinski-harabasz-index), [TF-IDF parameters](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html), [GroupShuffleSplit](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupShuffleSplit.html), [common pitfalls and leakage](https://scikit-learn.org/stable/common_pitfalls.html).

In [ ]:
source_column_audit.to_csv(OUTPUT_DIRECTORY / "all_18_source_fields_metadata_audit.csv", index=False, encoding="utf-8-sig")
source_validation_issues.to_csv(OUTPUT_DIRECTORY / "source_type_and_domain_validation_issues.csv", index=False, encoding="utf-8-sig")
internal_validation_performance.to_csv(OUTPUT_DIRECTORY / "eight_representations_grouped_internal_validation.csv", index=False, encoding="utf-8-sig")
all_eight_representation_out_of_fold_predictions.to_csv(OUTPUT_DIRECTORY / "all_eight_representations_every_out_of_fold_prediction.csv", index=False, encoding="utf-8-sig")
internal_representation_summary.to_csv(OUTPUT_DIRECTORY / "eight_representations_validation_ranking.csv", index=False, encoding="utf-8-sig")
external_holdout_performance.to_csv(OUTPUT_DIRECTORY / "eight_representations_frozen_holdout_metrics.csv", index=False, encoding="utf-8-sig")
external_holdout_predictions.to_csv(OUTPUT_DIRECTORY / "every_holdout_prediction_for_eight_representations.csv", index=False, encoding="utf-8-sig")
contextual_grouping_candidate_evaluations.to_csv(OUTPUT_DIRECTORY / "every_contextual_group_count_variance_balance_score.csv", index=False, encoding="utf-8-sig")
optimal_grouping_summary_for_display.to_csv(OUTPUT_DIRECTORY / "optimal_number_of_contextual_groups_by_variance_balance.csv", index=False, encoding="utf-8-sig")
observed_optimal_contextual_group_memberships.to_csv(OUTPUT_DIRECTORY / "observed_contextual_group_description_memberships.csv", index=False, encoding="utf-8-sig")


selected_representation_configuration = next(configuration for configuration in EVALUATED_OCCUPATIONAL_TEXT_REPRESENTATIONS if configuration["representation_name"] == internally_selected_representation_name)
selected_full_labeled_model = build_training_only_occupation_classification_pipeline(selected_representation_configuration)
selected_full_labeled_model.fit(all_labeled_model_inputs, all_labeled_occupation_codes)

partially_labeled_model_inputs = prepare_occupation_model_features(partially_labeled_participant_records)
suggested_codes_for_human_review = selected_full_labeled_model.predict(partially_labeled_model_inputs)

human_review_proposals = pd.DataFrame({
    "participant_id": partially_labeled_participant_records["ID"].astype(int).to_numpy(),
    "original_partially_labeled_code": partially_labeled_participant_records[OCCUPATION_CLASSIFICATION_TARGET_COLUMN].astype(str).to_numpy(),
    "unverified_model_suggested_code": suggested_codes_for_human_review,
    "operational_status": "mandatory_human_coder_review",
})

assert len(human_review_proposals) == 9
human_review_proposals.to_csv(OUTPUT_DIRECTORY / "nine_partially_labeled_cases_for_human_review.csv", index=False, encoding="utf-8-sig")


example_unseen_description_inputs = external_holdout_model_inputs.iloc[:2].copy()

for occupational_text_column in OCCUPATIONAL_DESCRIPTION_COLUMNS:
    example_unseen_description_inputs[occupational_text_column] = f"previously_unseen_occupation_description_{occupational_text_column}"

for representation_configuration in EVALUATED_OCCUPATIONAL_TEXT_REPRESENTATIONS:
    verified_model = fully_trained_classification_pipelines[representation_configuration["representation_name"]]
    assert len(verified_model.predict(example_unseen_description_inputs)) == 2


selected_holdout_predictions = external_holdout_predicted_codes_by_representation[internally_selected_representation_name]
selected_model_per_class_report = classification_report(external_holdout_occupation_codes, selected_holdout_predictions, zero_division=0, output_dict=True)
selected_model_per_class_table = pd.DataFrame(selected_model_per_class_report).T.drop(index=["accuracy", "macro avg", "weighted avg"], errors="ignore")
selected_model_per_class_table.index.name = "occupation_code"
selected_model_per_class_table.to_csv(OUTPUT_DIRECTORY / "internally_selected_model_per_class_holdout_metrics.csv", encoding="utf-8-sig")


verified_experiment_manifest = {
    "source_workbook_sha256": actual_source_workbook_sha256,
    "python_version": platform.python_version(),
    "scikit_learn_version": sklearn.__version__,
    "participant_records": len(participant_survey_records),
    "metadata_verified_columns": len(source_column_audit),
    "labeled_training_records": len(training_model_inputs),
    "external_holdout_records": len(external_holdout_model_inputs),
    "partially_labeled_human_review_records": len(partially_labeled_model_inputs),
    "candidate_representations": len(EVALUATED_OCCUPATIONAL_TEXT_REPRESENTATIONS),
    "candidate_contextual_grouping_configurations": 4,
    "candidate_contextual_group_count_evaluations": len(contextual_grouping_candidate_evaluations),
    "candidate_contextual_group_count_invalid_evaluations": int(contextual_grouping_candidate_evaluations["evaluation_status"].ne("valid").sum()),
    "optimal_grouping_selection_rule": "maximum finite Calinski-Harabasz score among all training-only candidate group counts; ties select fewer contextual groups",
    "contextual_grouping_selection_warning": "The score typically selects nearly one group per distinct description; mathematical score optimization does not prove useful semantic grouping",
    "classifier_selection_rule": "maximum mean macro-F1 on ten disjoint grouped cross-validation folds, with macro-recall and fixed complexity order breaking ties",
    "selected_representation_on_internal_validation": internally_selected_representation_name,
    "source_workbook_modified": False,
    "occupation_code_predictions_for_partial_labels_are_verified": False,
    "required_future_production_evidence": ["stable participant keys and version timestamps", "human-confirmed occupational coding labels", "time-aware and participant-grouped evaluation", "availability review for the final industry code", "semantic validation of contextual groups"],
}

(OUTPUT_DIRECTORY / "verified_experiment_manifest.json").write_text(json.dumps(verified_experiment_manifest, ensure_ascii=False, indent=2), encoding="utf-8")

print("All 18 source columns audited with declared metadata roles")
print("Eight representation pipelines evaluated on identical source and splits")
print("All contextual group-count scores exported without omitted candidates")
print("Nine partial occupation-code suggestions reserved for human review")
print("Original dataset is unchanged")
print("Internally selected classification representation:", internally_selected_representation_name)
display(human_review_proposals)

## Conclusion

**The classification system is a controlled prototype, not an approved production classifier.** Every source column has an explicit role, all eight representation variants use comparable evaluation conditions, and the contextual grouping findings are reported with their limitations. The presentation summarizes these same computed results without selecting models from external-holdout scores.

## Explicit model-family and hyperparameter comparison

After the established eight-representation pipeline and original prediction handoff, the same training data and the same ten grouped validation folds are used for **nine independently refitted classifier candidates** from three different estimator types: logistic regression, radial-basis support vector classification, and random forest. Each is fitted with the representation selected using training-only validation. Every candidate, fold, metric, and out-of-fold prediction is retained. This additive experiment does not overwrite the baseline training and prediction output.

The model family and its hyperparameter choice are selected only from training-fold evidence. Because 52 occupation codes appear in 149 training records and many codes have one example, unseen codes in validation folds create an unavoidable prediction limitation. Individual fold metrics and pooled out-of-fold metrics are displayed separately, and do not constitute an unbiased assessment of the entire selection procedure.

In [ ]:
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC


CLASSIFIER_HYPERPARAMETER_CANDIDATES = [
    {"classifier_family": "Logistic regression", "hyperparameter_configuration": "inverse_regularization_strength=0.2", "estimator": LogisticRegression(C=0.2, solver="lbfgs", max_iter=1200, random_state=REPRODUCIBLE_RANDOM_STATE)},
    {"classifier_family": "Logistic regression", "hyperparameter_configuration": "inverse_regularization_strength=1.0", "estimator": LogisticRegression(C=1.0, solver="lbfgs", max_iter=1200, random_state=REPRODUCIBLE_RANDOM_STATE)},
    {"classifier_family": "Logistic regression", "hyperparameter_configuration": "inverse_regularization_strength=5.0", "estimator": LogisticRegression(C=5.0, solver="lbfgs", max_iter=1200, random_state=REPRODUCIBLE_RANDOM_STATE)},
    {"classifier_family": "Support vector classification", "hyperparameter_configuration": "inverse_regularization_strength=0.3;radial_basis_kernel", "estimator": SVC(C=0.3, kernel="rbf", gamma="scale", probability=True, random_state=REPRODUCIBLE_RANDOM_STATE)},
    {"classifier_family": "Support vector classification", "hyperparameter_configuration": "inverse_regularization_strength=2.0;radial_basis_kernel", "estimator": SVC(C=2.0, kernel="rbf", gamma="scale", probability=True, random_state=REPRODUCIBLE_RANDOM_STATE)},
    {"classifier_family": "Support vector classification", "hyperparameter_configuration": "inverse_regularization_strength=8.0;radial_basis_kernel", "estimator": SVC(C=8.0, kernel="rbf", gamma="scale", probability=True, random_state=REPRODUCIBLE_RANDOM_STATE)},
    {"classifier_family": "Random forest", "hyperparameter_configuration": "number_of_trees=120;minimum_leaf_size=1", "estimator": RandomForestClassifier(n_estimators=120, min_samples_leaf=1, max_features="sqrt", n_jobs=1, random_state=REPRODUCIBLE_RANDOM_STATE)},
    {"classifier_family": "Random forest", "hyperparameter_configuration": "number_of_trees=120;minimum_leaf_size=2", "estimator": RandomForestClassifier(n_estimators=120, min_samples_leaf=2, max_features="sqrt", n_jobs=1, random_state=REPRODUCIBLE_RANDOM_STATE)},
    {"classifier_family": "Random forest", "hyperparameter_configuration": "number_of_trees=200;minimum_leaf_size=1", "estimator": RandomForestClassifier(n_estimators=200, min_samples_leaf=1, max_features="sqrt", n_jobs=1, random_state=REPRODUCIBLE_RANDOM_STATE)},
]

MODEL_SELECTION_PATHS = [
    {**configuration, "evaluation_path_identifier": f"model_configuration_{position:02d}"}
    for position, configuration in enumerate(CLASSIFIER_HYPERPARAMETER_CANDIDATES, start=1)
]


def build_classifier_model_candidate_on_selected_text_representation(model_configuration):
    original_training_only_pipeline = build_training_only_occupation_classification_pipeline(selected_representation_configuration)
    original_training_only_pipeline.set_params(four_digit_occupation_code_classifier=clone(model_configuration["estimator"]))

    return original_training_only_pipeline


all_classifier_hyperparameter_fold_scores = []
all_classifier_hyperparameter_out_of_fold_predictions = []
trained_fold_model_probabilities = {}

for validation_fold_number, (training_fold_positions, validation_fold_positions) in enumerate(internal_validation_group_partitions, start=1):
    fold_training_inputs = training_model_inputs.iloc[training_fold_positions]
    fold_validation_inputs = training_model_inputs.iloc[validation_fold_positions]
    fold_training_occupation_codes = training_occupation_codes.iloc[training_fold_positions]
    fold_validation_occupation_codes = training_occupation_codes.iloc[validation_fold_positions]
    fold_training_available_class_labels = np.unique(fold_training_occupation_codes.to_numpy())

    for model_configuration in MODEL_SELECTION_PATHS:
        path_identifier = model_configuration["evaluation_path_identifier"]
        candidate_classification_pipeline = build_classifier_model_candidate_on_selected_text_representation(model_configuration)
        training_start_time = perf_counter()

        candidate_classification_pipeline.fit(fold_training_inputs, fold_training_occupation_codes)
        predicted_validation_occupation_codes = candidate_classification_pipeline.predict(fold_validation_inputs)
        predicted_validation_class_probabilities = candidate_classification_pipeline.predict_proba(fold_validation_inputs)

        trained_fold_model_probabilities[(validation_fold_number, path_identifier)] = {
            "class_labels": candidate_classification_pipeline.classes_,
            "validation_row_positions": validation_fold_positions,
            "class_probabilities": predicted_validation_class_probabilities,
        }

        recorded_fold_evaluation = {
            "cross_validation_fold": validation_fold_number,
            "evaluation_path_identifier": path_identifier,
            "classifier_family": model_configuration["classifier_family"],
            "hyperparameter_configuration": model_configuration["hyperparameter_configuration"],
            "selected_text_representation": internally_selected_representation_name,
            "training_participant_records": len(fold_training_inputs),
            "validation_participant_records": len(fold_validation_inputs),
            "unseen_occupation_code_records": int((~fold_validation_occupation_codes.isin(fold_training_available_class_labels)).sum()),
            "training_elapsed_seconds": perf_counter() - training_start_time,
            **measure_occupation_classification_performance(fold_validation_occupation_codes, predicted_validation_occupation_codes),
        }

        all_classifier_hyperparameter_fold_scores.append(recorded_fold_evaluation)

        for source_row_position, true_occupation_code, predicted_occupation_code in zip(validation_fold_positions, fold_validation_occupation_codes, predicted_validation_occupation_codes):
            all_classifier_hyperparameter_out_of_fold_predictions.append({
                "cross_validation_fold": validation_fold_number,
                "evaluation_path_identifier": path_identifier,
                "source_row_position": int(source_row_position),
                "participant_id": int(completely_labeled_participant_records.iloc[training_row_positions[source_row_position]]["ID"]),
                "actual_occupation_code": str(true_occupation_code),
                "predicted_occupation_code": str(predicted_occupation_code),
            })

    print(f"Completed classifier optimization cross-validation fold {validation_fold_number} of {len(internal_validation_group_partitions)}", flush=True)


all_classifier_hyperparameter_fold_scores = pd.DataFrame(all_classifier_hyperparameter_fold_scores)
all_classifier_hyperparameter_out_of_fold_predictions = pd.DataFrame(all_classifier_hyperparameter_out_of_fold_predictions)

assert len(all_classifier_hyperparameter_fold_scores) == len(internal_validation_group_partitions) * len(MODEL_SELECTION_PATHS)
assert len(all_classifier_hyperparameter_out_of_fold_predictions) == len(training_model_inputs) * len(MODEL_SELECTION_PATHS)


classifier_hyperparameter_search_summary = (
    all_classifier_hyperparameter_fold_scores.groupby(["evaluation_path_identifier", "classifier_family", "hyperparameter_configuration"], as_index=False)
    .agg(mean_fold_macro_f1=("macro_f1", "mean"), standard_deviation_of_fold_macro_f1=("macro_f1", "std"), mean_fold_macro_recall=("macro_recall", "mean"), mean_fold_accuracy=("accuracy", "mean"), total_fitting_seconds=("training_elapsed_seconds", "sum"))
)

pooled_classifier_out_of_fold_scores = []

for model_configuration in MODEL_SELECTION_PATHS:
    path_identifier = model_configuration["evaluation_path_identifier"]
    model_predictions = all_classifier_hyperparameter_out_of_fold_predictions.loc[all_classifier_hyperparameter_out_of_fold_predictions["evaluation_path_identifier"].eq(path_identifier)]
    scores_across_all_validation_rows = measure_occupation_classification_performance(model_predictions["actual_occupation_code"], model_predictions["predicted_occupation_code"])
    pooled_classifier_out_of_fold_scores.append({"evaluation_path_identifier": path_identifier, **{f"pooled_out_of_fold_{metric}": value for metric, value in scores_across_all_validation_rows.items()}})

classifier_hyperparameter_search_summary = classifier_hyperparameter_search_summary.merge(pd.DataFrame(pooled_classifier_out_of_fold_scores), on="evaluation_path_identifier", validate="one_to_one")
classifier_hyperparameter_search_summary = classifier_hyperparameter_search_summary.sort_values(["classifier_family", "mean_fold_macro_f1", "mean_fold_macro_recall", "evaluation_path_identifier"], ascending=[True, False, False, True]).reset_index(drop=True)

optimized_individual_model_configurations = []

for classifier_family in ["Logistic regression", "Support vector classification", "Random forest"]:
    strongest_family_configuration = classifier_hyperparameter_search_summary.loc[classifier_hyperparameter_search_summary["classifier_family"].eq(classifier_family)].iloc[0]
    optimized_individual_model_configurations.append(next(configuration for configuration in MODEL_SELECTION_PATHS if configuration["evaluation_path_identifier"] == strongest_family_configuration["evaluation_path_identifier"]))

print("All nine explicitly enumerated classifier configurations, all ten folds:")
display(classifier_hyperparameter_search_summary.round(4))
print("Models independently selected using training-only validation:")
display(pd.DataFrame([{name: configuration[name] for name in ("classifier_family", "evaluation_path_identifier", "hyperparameter_configuration")} for configuration in optimized_individual_model_configurations]))

## Three-model probability voting and independent held-out comparison

All three tuned classifier types are retained in each soft-voting ensemble. **Four explicit probability weights** are evaluated on exactly the same ten validation partitions, using probabilities from independently fitted models from the relevant training fold. The candidate weights are selected using those training-fold scores only. Each selected classifier and all four ensemble weight paths are independently fitted on the original frozen training set and evaluated on the identical untouched 42-record holdout; these external scores do not change the chosen configuration.

The highest model probability is uncalibrated and must never be represented as verified confidence. The model suggestions for nine incompletely coded participant records require professional human review.

In [ ]:

SOFT_VOTING_ENSEMBLE_WEIGHT_CONFIGURATIONS = [
    {"ensemble_weight_path": "equal_probabilities", "model_weights": (1.0, 1.0, 1.0)},
    {"ensemble_weight_path": "double_logistic_regression", "model_weights": (2.0, 1.0, 1.0)},
    {"ensemble_weight_path": "double_support_vector", "model_weights": (1.0, 2.0, 1.0)},
    {"ensemble_weight_path": "double_random_forest", "model_weights": (1.0, 1.0, 2.0)},
]


optimized_model_order = [configuration["evaluation_path_identifier"] for configuration in optimized_individual_model_configurations]
all_soft_voting_ensemble_fold_scores = []
all_soft_voting_ensemble_out_of_fold_predictions = []

for validation_fold_number, (training_fold_positions, validation_fold_positions) in enumerate(internal_validation_group_partitions, start=1):
    current_fold_training_occupation_codes = training_occupation_codes.iloc[training_fold_positions]
    current_fold_validation_occupation_codes = training_occupation_codes.iloc[validation_fold_positions]
    fold_training_class_labels = np.unique(current_fold_training_occupation_codes.to_numpy())
    candidate_model_probability_matrices = []

    for optimized_model_identifier in optimized_model_order:
        trained_model_output = trained_fold_model_probabilities[(validation_fold_number, optimized_model_identifier)]
        assert np.array_equal(validation_fold_positions, trained_model_output["validation_row_positions"])
        assert np.array_equal(fold_training_class_labels, trained_model_output["class_labels"])
        candidate_model_probability_matrices.append(trained_model_output["class_probabilities"])

    for ensemble_configuration in SOFT_VOTING_ENSEMBLE_WEIGHT_CONFIGURATIONS:
        averaged_validation_class_probabilities = np.average(np.stack(candidate_model_probability_matrices, axis=0), axis=0, weights=np.asarray(ensemble_configuration["model_weights"]))
        predicted_validation_occupation_codes = fold_training_class_labels[np.argmax(averaged_validation_class_probabilities, axis=1)]

        all_soft_voting_ensemble_fold_scores.append({
            "cross_validation_fold": validation_fold_number,
            "ensemble_weight_path": ensemble_configuration["ensemble_weight_path"],
            "logistic_regression_weight": ensemble_configuration["model_weights"][0],
            "support_vector_weight": ensemble_configuration["model_weights"][1],
            "random_forest_weight": ensemble_configuration["model_weights"][2],
            "selected_text_representation": internally_selected_representation_name,
            "training_participant_records": len(training_fold_positions),
            "validation_participant_records": len(validation_fold_positions),
            **measure_occupation_classification_performance(current_fold_validation_occupation_codes, predicted_validation_occupation_codes),
        })

        for source_row_position, true_occupation_code, predicted_occupation_code in zip(validation_fold_positions, current_fold_validation_occupation_codes, predicted_validation_occupation_codes):
            all_soft_voting_ensemble_out_of_fold_predictions.append({
                "cross_validation_fold": validation_fold_number,
                "ensemble_weight_path": ensemble_configuration["ensemble_weight_path"],
                "source_row_position": int(source_row_position),
                "participant_id": int(completely_labeled_participant_records.iloc[training_row_positions[source_row_position]]["ID"]),
                "actual_occupation_code": str(true_occupation_code),
                "predicted_occupation_code": str(predicted_occupation_code),
            })


all_soft_voting_ensemble_fold_scores = pd.DataFrame(all_soft_voting_ensemble_fold_scores)
all_soft_voting_ensemble_out_of_fold_predictions = pd.DataFrame(all_soft_voting_ensemble_out_of_fold_predictions)

soft_voting_ensemble_weight_summary = (
    all_soft_voting_ensemble_fold_scores.groupby("ensemble_weight_path", as_index=False)
    .agg(mean_fold_macro_f1=("macro_f1", "mean"), standard_deviation_of_fold_macro_f1=("macro_f1", "std"), mean_fold_macro_recall=("macro_recall", "mean"), mean_fold_accuracy=("accuracy", "mean"))
)

ensemble_pooled_out_of_fold_scores = []

for ensemble_configuration in SOFT_VOTING_ENSEMBLE_WEIGHT_CONFIGURATIONS:
    ensemble_weight_path = ensemble_configuration["ensemble_weight_path"]
    ensemble_predictions = all_soft_voting_ensemble_out_of_fold_predictions.loc[all_soft_voting_ensemble_out_of_fold_predictions["ensemble_weight_path"].eq(ensemble_weight_path)]
    pooled_ensemble_scores = measure_occupation_classification_performance(ensemble_predictions["actual_occupation_code"], ensemble_predictions["predicted_occupation_code"])
    ensemble_pooled_out_of_fold_scores.append({"ensemble_weight_path": ensemble_weight_path, **{f"pooled_out_of_fold_{metric}": value for metric, value in pooled_ensemble_scores.items()}})

soft_voting_ensemble_weight_summary = soft_voting_ensemble_weight_summary.merge(pd.DataFrame(ensemble_pooled_out_of_fold_scores), on="ensemble_weight_path", validate="one_to_one")
soft_voting_ensemble_weight_summary = soft_voting_ensemble_weight_summary.sort_values(["mean_fold_macro_f1", "mean_fold_macro_recall", "ensemble_weight_path"], ascending=[False, False, True]).reset_index(drop=True)

selected_soft_voting_ensemble_weight_path = soft_voting_ensemble_weight_summary.iloc[0]["ensemble_weight_path"]
selected_soft_voting_ensemble_weight_configuration = next(configuration for configuration in SOFT_VOTING_ENSEMBLE_WEIGHT_CONFIGURATIONS if configuration["ensemble_weight_path"] == selected_soft_voting_ensemble_weight_path)


def build_soft_voting_ensemble_with_selected_representation(ensemble_configuration):
    selected_classifier_components = [
        ("logistic_regression", clone(optimized_individual_model_configurations[0]["estimator"])),
        ("support_vector_classification", clone(optimized_individual_model_configurations[1]["estimator"])),
        ("random_forest", clone(optimized_individual_model_configurations[2]["estimator"])),
    ]

    classification_pipeline = build_training_only_occupation_classification_pipeline(selected_representation_configuration)
    classification_pipeline.set_params(four_digit_occupation_code_classifier=VotingClassifier(estimators=selected_classifier_components, voting="soft", weights=ensemble_configuration["model_weights"], n_jobs=1))

    return classification_pipeline


final_model_and_ensemble_options = [
    {"evaluation_path_identifier": configuration["evaluation_path_identifier"], "model_family": configuration["classifier_family"], "build_pipeline": lambda configuration=configuration: build_classifier_model_candidate_on_selected_text_representation(configuration)}
    for configuration in optimized_individual_model_configurations
] + [
    {"evaluation_path_identifier": configuration["ensemble_weight_path"], "model_family": "Three-model soft-voting ensemble", "build_pipeline": lambda configuration=configuration: build_soft_voting_ensemble_with_selected_representation(configuration)}
    for configuration in SOFT_VOTING_ENSEMBLE_WEIGHT_CONFIGURATIONS
]


all_tuned_model_and_ensemble_holdout_scores = []
all_tuned_model_and_ensemble_holdout_predictions = []
trained_ensemble_candidates_on_frozen_training = {}

for final_model_option in final_model_and_ensemble_options:
    final_option_identifier = final_model_option["evaluation_path_identifier"]
    fitted_final_option = final_model_option["build_pipeline"]()
    final_training_start_time = perf_counter()
    fitted_final_option.fit(training_model_inputs, training_occupation_codes)
    final_option_predictions = fitted_final_option.predict(external_holdout_model_inputs)

    trained_ensemble_candidates_on_frozen_training[final_option_identifier] = fitted_final_option
    all_tuned_model_and_ensemble_holdout_scores.append({
        "evaluation_path_identifier": final_option_identifier,
        "model_family": final_model_option["model_family"],
        "correctly_classified_participant_records": int(np.sum(final_option_predictions == external_holdout_occupation_codes.to_numpy())),
        "evaluated_holdout_records": len(external_holdout_model_inputs),
        "training_elapsed_seconds": perf_counter() - final_training_start_time,
        "selected_ensemble_weights_based_on_training_only": final_option_identifier == selected_soft_voting_ensemble_weight_path,
        **measure_occupation_classification_performance(external_holdout_occupation_codes, final_option_predictions),
    })

    for actual_participant_id, true_occupation_code, predicted_occupation_code in zip(completely_labeled_participant_records.iloc[external_holdout_row_positions]["ID"], external_holdout_occupation_codes, final_option_predictions):
        all_tuned_model_and_ensemble_holdout_predictions.append({
            "evaluation_path_identifier": final_option_identifier,
            "participant_id": int(actual_participant_id),
            "actual_occupation_code": str(true_occupation_code),
            "predicted_occupation_code": str(predicted_occupation_code),
        })


all_tuned_model_and_ensemble_holdout_scores = pd.DataFrame(all_tuned_model_and_ensemble_holdout_scores)
all_tuned_model_and_ensemble_holdout_predictions = pd.DataFrame(all_tuned_model_and_ensemble_holdout_predictions)

assert len(all_tuned_model_and_ensemble_holdout_predictions) == len(external_holdout_model_inputs) * len(final_model_and_ensemble_options)


trained_final_ensemble_on_all_labeled_records = build_soft_voting_ensemble_with_selected_representation(selected_soft_voting_ensemble_weight_configuration)
trained_final_ensemble_on_all_labeled_records.fit(all_labeled_model_inputs, all_labeled_occupation_codes)
ensemble_predictions_for_partially_labeled_records = trained_final_ensemble_on_all_labeled_records.predict(partially_labeled_model_inputs)
ensemble_class_probabilities_for_partially_labeled_records = trained_final_ensemble_on_all_labeled_records.predict_proba(partially_labeled_model_inputs)

final_ensemble_human_review_suggestions = pd.DataFrame({
    "participant_id": partially_labeled_participant_records["ID"].astype(int).to_numpy(),
    "original_partially_labeled_occupation_code": partially_labeled_participant_records[OCCUPATION_CLASSIFICATION_TARGET_COLUMN].astype(str).to_numpy(),
    "unverified_ensemble_suggested_occupation_code": ensemble_predictions_for_partially_labeled_records,
    "uncalibrated_maximum_predicted_class_probability": ensemble_class_probabilities_for_partially_labeled_records.max(axis=1),
    "operational_status": "mandatory_human_coder_review",
})

assert len(final_ensemble_human_review_suggestions) == 9

print("All four ensemble weight paths, including weaker alternatives:")
display(soft_voting_ensemble_weight_summary.round(4))
print("Untouched held-out test set, frozen after training-only selection:")
display(all_tuned_model_and_ensemble_holdout_scores.round(4))
print("Nine additional ensemble proposals are unverified and require human review:")
display(final_ensemble_human_review_suggestions)

## All experimental paths · complete, unfiltered evidence and graphical comparisons

The output directory contains every source-column assessment, all contextual-group candidate scores, all representation-fold scores and predictions, every classifier hyperparameter fold and prediction, every soft-voting weighting fold and prediction, and all external-holdout comparisons. No experiment is hidden because of lower performance. Curves, bar charts and per-fold heat maps provide both global comparison and the individual path traces.

The original 10-slide assignment limit applies to the presentation, not to the notebook or the CSV evidence archive. The presentation shows the most decision-relevant comparisons, while the notebook and exported tables retain the complete results.

In [ ]:

verified_v6_tables = {
    "all_18_source_fields_metadata_audit.csv": source_column_audit,
    "source_type_and_domain_validation_issues.csv": source_validation_issues,
    "all_eight_representations_ten_fold_metrics.csv": internal_validation_performance,
    "all_eight_representations_ten_fold_predictions.csv": all_eight_representation_out_of_fold_predictions,
    "eight_representations_ten_fold_summary.csv": internal_representation_summary,
    "every_contextual_group_count_variance_balance_score.csv": contextual_grouping_candidate_evaluations,
    "optimal_number_of_contextual_groups_by_variance_balance.csv": optimal_grouping_summary_for_display,
    "every_classifier_hyperparameter_ten_fold_score.csv": all_classifier_hyperparameter_fold_scores,
    "every_classifier_hyperparameter_ten_fold_prediction.csv": all_classifier_hyperparameter_out_of_fold_predictions,
    "every_classifier_hyperparameter_summary.csv": classifier_hyperparameter_search_summary,
    "every_ensemble_weight_ten_fold_score.csv": all_soft_voting_ensemble_fold_scores,
    "every_ensemble_weight_ten_fold_prediction.csv": all_soft_voting_ensemble_out_of_fold_predictions,
    "every_ensemble_weight_summary.csv": soft_voting_ensemble_weight_summary,
    "every_selected_classifier_and_ensemble_heldout_score.csv": all_tuned_model_and_ensemble_holdout_scores,
    "every_selected_classifier_and_ensemble_heldout_prediction.csv": all_tuned_model_and_ensemble_holdout_predictions,
    "final_ensemble_nine_suggestions_for_human_review.csv": final_ensemble_human_review_suggestions,
}

for output_filename, output_table in verified_v6_tables.items():
    output_table.to_csv(OUTPUT_DIRECTORY / output_filename, index=False, encoding="utf-8-sig")


representations_display_order = [configuration["representation_name"] for configuration in EVALUATED_OCCUPATIONAL_TEXT_REPRESENTATIONS]
representation_short_names = [name.split(" · ", maxsplit=1)[-1] for name in representations_display_order]
representation_summary_for_graph = internal_representation_summary.set_index("representation_name").loc[representations_display_order]
representation_metric_matrix = internal_validation_performance.pivot(index="representation_name", columns="internal_validation_partition", values="macro_f1").loc[representations_display_order]

representation_figure, representation_axes = plt.subplots(1, 2, figsize=(18, 7.8), gridspec_kw={"width_ratios": [1, 1.15]})
representation_display_positions = np.arange(len(representation_short_names))
representation_axes[0].barh(representation_display_positions, representation_summary_for_graph["internal_macro_f1_mean"], xerr=representation_summary_for_graph["internal_macro_f1_standard_deviation"], color="#0E7490", capsize=3)
representation_axes[0].set_yticks(representation_display_positions, representation_short_names, fontsize=9)
representation_axes[0].invert_yaxis()
representation_axes[0].set_xlabel("Mean ten-fold grouped-validation macro-F1 ± fold standard deviation")
representation_axes[0].set_title("Every text-representation path")
representation_axes[0].set_xlim(0, max(0.65, float(representation_summary_for_graph["internal_macro_f1_mean"].max()) + 0.12))
representation_axes[0].grid(axis="x", alpha=0.15)

for representation_position, representation_score in enumerate(representation_summary_for_graph["internal_macro_f1_mean"]):
    representation_axes[0].text(float(representation_score) + 0.004, representation_position, f"{representation_score:.3f}", va="center", fontsize=9)

representation_heatmap = representation_axes[1].imshow(representation_metric_matrix.to_numpy(), cmap="Blues", vmin=0, vmax=1, aspect="auto")
representation_axes[1].set_xticks(np.arange(10), [str(fold_number) for fold_number in range(1, 11)])
representation_axes[1].set_yticks(np.arange(len(representation_short_names)), representation_short_names, fontsize=9)
representation_axes[1].set_xlabel("Grouped cross-validation fold number")
representation_axes[1].set_title("Every validation score, every fold")

for text_row in range(representation_metric_matrix.shape[0]):
    for text_column in range(representation_metric_matrix.shape[1]):
        observed_f1 = representation_metric_matrix.iloc[text_row, text_column]
        representation_axes[1].text(text_column, text_row, f"{observed_f1:.2f}", ha="center", va="center", color="white" if observed_f1 >= 0.55 else "#09202B", fontsize=7)

representation_figure.colorbar(representation_heatmap, ax=representation_axes[1], fraction=0.02, pad=0.01)
representation_figure.suptitle("Eight representation paths · one unchanged transformation choice for all six text columns", fontsize=15, weight="bold")
representation_figure.tight_layout()
representation_figure.savefig(OUTPUT_DIRECTORY / "ten_fold_eight_representation_full_comparison.png", dpi=185, bbox_inches="tight")
plt.show()


classifier_family_colors = {"Logistic regression": "#0284C7", "Support vector classification": "#F59E0B", "Random forest": "#059669"}
classifier_summary_for_chart = classifier_hyperparameter_search_summary.sort_values("evaluation_path_identifier")
classifier_figure, classifier_axes = plt.subplots(1, 2, figsize=(18, 7.8), gridspec_kw={"width_ratios": [1, 1.1]})
classifier_display_positions = np.arange(len(classifier_summary_for_chart))

classifier_axes[0].barh(classifier_display_positions, classifier_summary_for_chart["mean_fold_macro_f1"], xerr=classifier_summary_for_chart["standard_deviation_of_fold_macro_f1"], color=[classifier_family_colors[family] for family in classifier_summary_for_chart["classifier_family"]], capsize=3)
classifier_axes[0].invert_yaxis()
classifier_axes[0].set_yticks(classifier_display_positions, [f"{identifier} | {family}" for identifier, family in zip(classifier_summary_for_chart["evaluation_path_identifier"], classifier_summary_for_chart["classifier_family"])], fontsize=9)
classifier_axes[0].set_xlabel("Mean ten-fold macro-F1 ± fold standard deviation")
classifier_axes[0].set_title("Every classifier hyperparameter path")
classifier_axes[0].grid(axis="x", alpha=0.2)

for classifier_position, classifier_score in enumerate(classifier_summary_for_chart["mean_fold_macro_f1"]):
    classifier_axes[0].text(float(classifier_score) + 0.002, classifier_position, f"{classifier_score:.3f}", va="center", fontsize=8)

classifier_metric_matrix = all_classifier_hyperparameter_fold_scores.pivot(index="evaluation_path_identifier", columns="cross_validation_fold", values="macro_f1").loc[classifier_summary_for_chart["evaluation_path_identifier"]]
classifier_heatmap = classifier_axes[1].imshow(classifier_metric_matrix.to_numpy(), cmap="YlGnBu", vmin=0, vmax=1, aspect="auto")
classifier_axes[1].set_yticks(np.arange(len(classifier_metric_matrix)), classifier_metric_matrix.index, fontsize=9)
classifier_axes[1].set_xticks(np.arange(10), [str(fold_number) for fold_number in range(1, 11)])
classifier_axes[1].set_xlabel("Grouped cross-validation fold number")
classifier_axes[1].set_title("Every classifier score in every fold")

for text_row in range(classifier_metric_matrix.shape[0]):
    for text_column in range(classifier_metric_matrix.shape[1]):
        observed_f1 = classifier_metric_matrix.iloc[text_row, text_column]
        classifier_axes[1].text(text_column, text_row, f"{observed_f1:.2f}", ha="center", va="center", color="white" if observed_f1 >= 0.55 else "#09202B", fontsize=7)

classifier_figure.colorbar(classifier_heatmap, ax=classifier_axes[1], fraction=0.02, pad=0.01)
classifier_figure.suptitle("Three distinct classifier types · all nine hyperparameter configurations", fontsize=15, weight="bold")
classifier_figure.tight_layout()
classifier_figure.savefig(OUTPUT_DIRECTORY / "three_classifiers_nine_hyperparameter_paths_ten_fold_graph.png", dpi=185, bbox_inches="tight")
plt.show()


ensemble_weight_summary_for_chart = soft_voting_ensemble_weight_summary.sort_values("ensemble_weight_path")
ensemble_fold_metric_matrix = all_soft_voting_ensemble_fold_scores.pivot(index="ensemble_weight_path", columns="cross_validation_fold", values="macro_f1").loc[ensemble_weight_summary_for_chart["ensemble_weight_path"]]
ensemble_figure, ensemble_axes = plt.subplots(1, 2, figsize=(16, 5.5), gridspec_kw={"width_ratios": [1, 1.25]})
ensemble_axes[0].barh(ensemble_weight_summary_for_chart["ensemble_weight_path"], ensemble_weight_summary_for_chart["mean_fold_macro_f1"], xerr=ensemble_weight_summary_for_chart["standard_deviation_of_fold_macro_f1"], color="#64748B", capsize=3)
ensemble_axes[0].invert_yaxis()
ensemble_axes[0].set_xlabel("Mean ten-fold grouped-validation macro-F1")
ensemble_axes[0].set_title("All four ensemble weighting paths")
ensemble_axes[0].grid(axis="x", alpha=0.15)

for ensemble_position, ensemble_score in enumerate(ensemble_weight_summary_for_chart["mean_fold_macro_f1"]):
    ensemble_axes[0].text(float(ensemble_score) + 0.003, ensemble_position, f"{ensemble_score:.3f}", va="center", fontsize=9)

ensemble_heatmap = ensemble_axes[1].imshow(ensemble_fold_metric_matrix.to_numpy(), cmap="Purples", vmin=0, vmax=1, aspect="auto")
ensemble_axes[1].set_yticks(np.arange(len(ensemble_fold_metric_matrix)), ensemble_fold_metric_matrix.index, fontsize=9)
ensemble_axes[1].set_xticks(np.arange(10), [str(fold_number) for fold_number in range(1, 11)])
ensemble_axes[1].set_title("All ensemble weighting paths by fold")
ensemble_axes[1].set_xlabel("Grouped cross-validation fold number")

for text_row in range(ensemble_fold_metric_matrix.shape[0]):
    for text_column in range(ensemble_fold_metric_matrix.shape[1]):
        observed_f1 = ensemble_fold_metric_matrix.iloc[text_row, text_column]
        ensemble_axes[1].text(text_column, text_row, f"{observed_f1:.2f}", ha="center", va="center", color="white" if observed_f1 >= 0.55 else "#24102B", fontsize=8)

ensemble_figure.colorbar(ensemble_heatmap, ax=ensemble_axes[1], fraction=0.03, pad=0.01)
ensemble_figure.tight_layout()
ensemble_figure.savefig(OUTPUT_DIRECTORY / "four_soft_voting_ensemble_weight_paths_ten_fold_graph.png", dpi=185, bbox_inches="tight")
plt.show()


holdout_figure, holdout_axis = plt.subplots(figsize=(16, 6.5))
heldout_table_for_graph = all_tuned_model_and_ensemble_holdout_scores.sort_values("evaluation_path_identifier")
model_path_display_positions = np.arange(len(heldout_table_for_graph))

holdout_axis.bar(model_path_display_positions - 0.18, heldout_table_for_graph["macro_f1"], width=0.36, label="Held-out macro-F1", color="#0E7490")
holdout_axis.bar(model_path_display_positions + 0.18, heldout_table_for_graph["macro_recall"], width=0.36, label="Held-out macro-recall", color="#F59E0B")
holdout_axis.set_xticks(model_path_display_positions, heldout_table_for_graph["evaluation_path_identifier"], rotation=15, ha="right")
holdout_axis.set_ylabel("External holdout metrics, descriptive comparison only")
holdout_axis.set_ylim(0, 1)
holdout_axis.set_title("Every selected model and every ensemble weighting path · same 42-record untouched holdout", weight="bold")
holdout_axis.grid(axis="y", alpha=0.15)
holdout_axis.legend(frameon=False)

for observed_bar in holdout_axis.patches:
    holdout_axis.annotate(f"{observed_bar.get_height():.3f}", (observed_bar.get_x() + observed_bar.get_width() / 2, observed_bar.get_height()), xytext=(0, 4), textcoords="offset points", ha="center", fontsize=8)

holdout_figure.tight_layout()
holdout_figure.savefig(OUTPUT_DIRECTORY / "three_classifiers_four_ensembles_external_holdout_comparison.png", dpi=185, bbox_inches="tight")
plt.show()


validation_score_by_classifier_and_ensemble = pd.concat([
    classifier_hyperparameter_search_summary.assign(path_family="Individual classifier").rename(columns={"evaluation_path_identifier": "evaluation_path"})[["evaluation_path", "path_family", "mean_fold_macro_f1", "pooled_out_of_fold_macro_f1"]],
    soft_voting_ensemble_weight_summary.assign(path_family="Soft-voting ensemble").rename(columns={"ensemble_weight_path": "evaluation_path"})[["evaluation_path", "path_family", "mean_fold_macro_f1", "pooled_out_of_fold_macro_f1"]],
], ignore_index=True)

training_only_selection_comparison_figure, training_only_selection_comparison_axis = plt.subplots(figsize=(14, 7.0))
training_only_selection_comparison_table = validation_score_by_classifier_and_ensemble.sort_values("mean_fold_macro_f1")
training_only_selection_comparison_axis.barh(training_only_selection_comparison_table["evaluation_path"], training_only_selection_comparison_table["mean_fold_macro_f1"], color=np.where(training_only_selection_comparison_table["path_family"].eq("Soft-voting ensemble"), "#B45309", "#0369A1"))
training_only_selection_comparison_axis.set_xlabel("Training-only grouped ten-fold mean macro-F1")
training_only_selection_comparison_axis.set_title("Every individual hyperparameter path and every ensemble path · common evaluation", weight="bold")
training_only_selection_comparison_axis.grid(axis="x", alpha=0.2)

for observed_bar in training_only_selection_comparison_axis.patches:
    training_only_selection_comparison_axis.text(observed_bar.get_width() + 0.003, observed_bar.get_y() + observed_bar.get_height() / 2, f"{observed_bar.get_width():.3f}", va="center", fontsize=8)

training_only_selection_comparison_figure.tight_layout()
training_only_selection_comparison_figure.savefig(OUTPUT_DIRECTORY / "all_classifier_and_ensemble_training_validation_paths.png", dpi=185, bbox_inches="tight")
plt.show()


cross_validation_observation_manifest = {
    "source_workbook_sha256": actual_source_workbook_sha256,
    "scikit_learn_version": sklearn.__version__,
    "training_records": len(training_model_inputs),
    "heldout_records": len(external_holdout_model_inputs),
    "training_description_group_count": training_description_groups.nunique(),
    "disjoint_grouped_validation_folds": len(internal_validation_group_partitions),
    "all_eight_representation_fold_evaluations": len(internal_validation_performance),
    "all_eight_representation_out_of_fold_predictions": len(all_eight_representation_out_of_fold_predictions),
    "fully_evaluated_classifier_hyperparameter_paths": len(MODEL_SELECTION_PATHS),
    "all_classifier_fold_evaluations": len(all_classifier_hyperparameter_fold_scores),
    "all_classifier_out_of_fold_predictions": len(all_classifier_hyperparameter_out_of_fold_predictions),
    "fully_evaluated_soft_voting_weight_paths": len(SOFT_VOTING_ENSEMBLE_WEIGHT_CONFIGURATIONS),
    "all_ensemble_fold_evaluations": len(all_soft_voting_ensemble_fold_scores),
    "all_ensemble_out_of_fold_predictions": len(all_soft_voting_ensemble_out_of_fold_predictions),
    "selected_text_representation_on_training_only": internally_selected_representation_name,
    "optimized_individual_classifiers_on_training_only": [{"classifier_family": configuration["classifier_family"], "hyperparameter_configuration": configuration["hyperparameter_configuration"]} for configuration in optimized_individual_model_configurations],
    "selected_soft_voting_ensemble_weight_path_on_training_only": selected_soft_voting_ensemble_weight_path,
    "training_criterion": "mean macro-F1 across ten disjoint grouped folds, macro-recall tie-breaking",
    "evaluation_warning": "Many occupation labels have only one observation; some labels are absent from training folds and cannot be predicted; mean fold metrics are unstable and optimistic after tuning",
    "ensemble_probability_warning": "Soft-voting inputs are not proven calibrated; maximum predicted probability is not a verified confidence level",
    "source_workbook_modified": False,
    "previous_notebook_versions_modified": False,
    "all_paths_exported": sorted(verified_v6_tables),
}

(OUTPUT_DIRECTORY / "v6_complete_experimental_evidence_manifest.json").write_text(json.dumps(cross_validation_observation_manifest, ensure_ascii=False, indent=2), encoding="utf-8")

with pd.option_context("display.max_rows", 150, "display.max_columns", None, "display.width", 240):
    print("All nine classifier hyperparameters and their scores:")
    display(classifier_hyperparameter_search_summary.round(4))
    print("All four ensemble paths and their scores:")
    display(soft_voting_ensemble_weight_summary.round(4))
    print("All final held-out scores, including alternatives not selected:")
    display(all_tuned_model_and_ensemble_holdout_scores.round(4))

print("Complete fold-level scores and predictions for all paths have been exported without filtering")

## Source and verification notes

- [GroupKFold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupKFold.html) uses disjoint groups and produces ten folds.
- [VotingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.VotingClassifier.html) averages component probabilities for soft voting.
- [Support Vector Classification](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html) estimates probabilities using additional internal fitting and may yield unreliable probability values on small datasets.
- [Nested versus non-nested validation](https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html) explains why tuning scores can be optimistic and why the held-out evaluation must remain separate.
- The dataset is simulated and its 90% manual coding statement is background context rather than independently verified ground truth. The classification approach is a supervised prototype, not a production-approved coding service.

**Reproducibility:** Original workbook content and previous notebook versions are unmodified. The SHA-256 input contract is checked before execution. Every learned representation and estimator uses only the corresponding training partition.